# 第十一周：当前队列内部验证与全量敏感性

独立小鼠为重复单位；LOMO、全局按年龄分层的配对鼠级 bootstrap B=500；HVG2000/3000/5000、PC7/10/20/30、细胞数门槛和均值/双侧各10%截尾均值。所有设置与失败均保留。固定PCA基内中心交叉拟合属于条件内部验证，不宣称完全嵌套HVG/PCA验证或外部验证。


In [1]:
import json
import hashlib
import importlib.metadata
import warnings
import gc
import h5py
import anndata as ad
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy.stats import spearmanr, trim_mean
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import squareform
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42
SEED=2026
NODES=["Kidney | lymphoid","Kidney | myeloid","Liver | lymphoid","Liver | myeloid","Lung | lymphoid","Lung | myeloid","spleen/marrow | lymphoid","spleen/marrow | myeloid"]

def axis_cos(a,b):
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den>1e-12 else np.nan

def fit_axis(frame,k=7,mode="centroid",weights=None):
    cols=[f"PC{i+1}" for i in range(k)]
    s=frame.copy()
    w=np.ones(len(s)) if weights is None else s.mouse_id.map(weights).fillna(0).to_numpy(float)
    keep=w>0;s=s.loc[keep].copy();w=w[keep]
    sizes=s.age_group.value_counts()
    ny,no=int(sizes.get("young",0)),int(sizes.get("old",0))
    result={"n_young":ny,"n_old":no,"n_middle":int(sizes.get("middle",0)),"status":"not_estimable","reason":"","formal":ny>=2 and no>=2}
    if not ny or not no:
        result["reason"]="missing_young_or_old";return result
    y=s.age_group.eq("young").to_numpy();o=s.age_group.eq("old").to_numpy()
    X=s[cols].to_numpy(float)
    mu_y=np.average(X[y],axis=0,weights=w[y])
    d=np.average(X[o],axis=0,weights=w[o])-mu_y
    if mode=="sex_adjusted":
        terms=[np.ones(len(s))]
        if s.sex.nunique()>1:terms.append(s.sex.eq("female").to_numpy(float))
        if s.age_group.eq("middle").any():terms.append(s.age_group.eq("middle").to_numpy(float))
        terms.append(o.astype(float))
        design=np.column_stack(terms);rootw=np.sqrt(w)
        rank=np.linalg.matrix_rank(design*rootw[:,None])
        result.update(design_rank=int(rank),design_columns=design.shape[1],residual_df=int(len(s)-rank))
        if rank<design.shape[1] or len(s)<=rank:
            result["reason"]="rank_deficient_or_no_residual_df";return result
        coef=np.linalg.lstsq(design*rootw[:,None],X*rootw[:,None],rcond=None)[0]
        d=coef[-1];mu_y=coef[0]
        result["sex_coefficient"]=coef[1] if s.sex.nunique()>1 else np.zeros(k)
    norm=float(np.linalg.norm(d))
    if not np.isfinite(norm) or norm<=1e-12:
        result["reason"]="zero_or_nonfinite_direction";return result
    result.update(d=d,mu_y=mu_y,norm=norm,status="complete" if result["formal"] else "exploratory_low_replication")
    return result

def project_axis(frame,fit,k):
    cols=[f"PC{i+1}" for i in range(k)]
    if "d" not in fit:return np.full(len(frame),np.nan)
    X=frame[cols].to_numpy(float)
    # Standardise held-out PCs to male reference using training-only sex coefficients.
    if "sex_coefficient" in fit:X=X-frame.sex.eq("female").to_numpy(float)[:,None]*fit["sex_coefficient"]
    return (X-fit["mu_y"])@fit["d"]/fit["norm"]

def describe_age(scores,ages):
    s=np.asarray(scores,float);a=np.asarray(ages,float);valid=np.isfinite(s)&np.isfinite(a)
    return float(spearmanr(s[valid],a[valid]).statistic) if valid.sum()>=4 and np.unique(s[valid]).size>1 and np.unique(a[valid]).size>1 else np.nan

def aggregate_coordinates(obs,coords,aggregation="mean",cell_mask=None,cap=None,seed=2026):
    keep=np.ones(len(obs),bool) if cell_mask is None else np.asarray(cell_mask,bool)
    sub=obs.loc[keep].copy().reset_index(drop=True)
    z=np.asarray(coords)[keep]
    groups=sub.groupby(["mouse_id","tissue","mct"],observed=True,sort=True).indices
    rng=np.random.default_rng(seed);rows=[]
    for key,idx in groups.items():
        if cap is not None and len(idx)>cap:idx=rng.choice(idx,size=cap,replace=False)
        block=z[idx]
        vector=block.mean(0) if aggregation=="mean" else trim_mean(block,.1,axis=0)
        meta=sub.iloc[idx[0]]
        rows.append(dict(mouse_id=str(key[0]),tissue=str(key[1]),mct=str(key[2]),node=str(key[1])+" | "+str(key[2]),age=int(meta.age),age_group=str(meta.age_group),sex=str(meta.sex),n_cells=len(idx),**{f"PC{i+1}":float(v) for i,v in enumerate(vector)}))
    return pd.DataFrame(rows)

def cohort_filter(table,keys,min_cells):
    valid=table.apply(lambda r:(r.mouse_id,r.tissue,r.mct) in keys,axis=1)
    return table.loc[valid & table.n_cells.ge(min_cells)].copy()

def evaluate_axes(table,k=7,mode="centroid"):
    fits={};summ=[];fullscores=[];lomo=[]
    for node in NODES:
        g=table.loc[table.node.eq(node)].copy()
        fit=fit_axis(g,k,mode);fits[node]=fit
        row={key:value for key,value in fit.items() if key not in ["d","mu_y","sex_coefficient"]}
        row.update(node=node,n_mice=len(g),n_cells=int(g.n_cells.sum()) if len(g) else 0,median_cells=float(g.n_cells.median()) if len(g) else np.nan)
        if "d" in fit:
            gs=g.copy();gs["in_sample_score"]=project_axis(g,fit,k);fullscores.append(gs)
            for mouse in g.mouse_id:
                train=g.loc[g.mouse_id.ne(mouse)]
                held=g.loc[g.mouse_id.eq(mouse)];lf=fit_axis(train,k,mode)
                item=dict(node=node,mouse_id=mouse,age=int(held.age.iloc[0]),age_group=held.age_group.iloc[0],sex=held.sex.iloc[0],status=lf["status"],reason=lf["reason"],n_train_young=lf["n_young"],n_train_old=lf["n_old"],score=float(project_axis(held,lf,k)[0]),cosine_to_full=axis_cos(fit["d"],lf["d"]) if "d" in lf else np.nan)
                lomo.append(item)
            items=[x for x in lomo if x["node"]==node]
            row["rho_age_crossfit"]=describe_age([r["score"] for r in items],[r["age"] for r in items])
            vals=[r["cosine_to_full"] for r in items if np.isfinite(r["cosine_to_full"])]
            row["min_lomo_cosine"]=min(vals) if vals else np.nan
            row["median_lomo_cosine"]=float(np.median(vals)) if vals else np.nan
            row["n_lomo_estimable"]=len(vals)
        summ.append(row)
    eligible=[n for n in NODES if fits[n]["formal"] and "d" in fits[n]]
    cosine=pd.DataFrame([[axis_cos(fits[a]["d"],fits[b]["d"]) for b in eligible] for a in eligible],index=eligible,columns=eligible)
    return {"fits":fits,"summary":pd.DataFrame(summ),"scores":pd.concat(fullscores,ignore_index=True) if fullscores else pd.DataFrame(),"lomo":pd.DataFrame(lomo),"cosine":cosine,"eligible":eligible}

def compare_scenarios(reference,trial):
    common=[n for n in reference["eligible"] if n in trial["eligible"]]
    corr=np.nan;sign=np.nan;maxdiff=np.nan
    if len(common)>=3:
        idx=np.triu_indices(len(common),1)
        a=reference["cosine"].loc[common,common].to_numpy()[idx]
        b=trial["cosine"].loc[common,common].to_numpy()[idx]
        corr=describe_age(b,a)
        sign=float(np.mean(np.sign(a)==np.sign(b)));maxdiff=float(np.max(abs(a-b)))
    return dict(n_common_nodes=len(common),cosine_matrix_spearman=corr,pair_sign_agreement=sign,max_pair_cosine_change=maxdiff)

def full_bootstrap(table,k=7,B=500,seed=2026):
    base=evaluate_axes(table,k);mice=table[["mouse_id","age_group","sex"]].drop_duplicates("mouse_id")
    rng=np.random.default_rng(seed);drawrows=[];pairrows=[]
    groups={n:table.loc[table.node.eq(n)] for n in NODES}
    for b in range(B):
        draws=[]
        for age in ["young","middle","old"]:
            pool=mice.loc[mice.age_group.eq(age),"mouse_id"].to_numpy()
            if len(pool):draws.extend(rng.choice(pool,len(pool),replace=True))
        weights=pd.Series(draws).value_counts().to_dict();fits={}
        for n in base["eligible"]:
            fit=fit_axis(groups[n],k,weights=weights);fits[n]=fit
            drawrows.append(dict(iteration=b,node=n,status=fit["status"],reason=fit["reason"],n_unique_young=fit["n_young"],n_unique_old=fit["n_old"],direction_norm=fit.get("norm",np.nan),cosine_to_baseline=axis_cos(base["fits"][n]["d"],fit["d"]) if "d" in fit else np.nan))
        for i,a in enumerate(base["eligible"]):
            for c in base["eligible"][i+1:]:
                value=axis_cos(fits[a]["d"],fits[c]["d"]) if "d" in fits[a] and "d" in fits[c] else np.nan
                pairrows.append(dict(iteration=b,node1=a,node2=c,cosine=value))
    return pd.DataFrame(drawrows),pd.DataFrame(pairrows)


## 1. HVG与PCA表示敏感性

严格复用06的年龄无关seurat_v3排序和原始全注释细胞拟合范围，不因结果选择基因。重新拟合PCA；不同基之间仅比较基无关的得分、相似性及其稳定性。


In [2]:
from sklearn.decomposition import PCA
import time
! powershell.exe -NoProfile -Command "New-Item -ItemType Directory -Force '../results/sensitivity/current_cohort_20261030' | Out-Null"
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    hvg_final_obs=read_elem(handle["obs"])
    hvg_final_var=read_elem(handle["var"])
    hvg_frozen_pca=np.asarray(read_elem(handle["obsm/X_pca"]))
with h5py.File("../data_interim/cell_type_annotated.h5ad","r") as handle:
    hvg_obs=read_elem(handle["obs"])
    hvg_var=read_elem(handle["var"])
assert hvg_obs.index.is_unique
hvg_positions=hvg_obs.index.get_indexer(hvg_final_obs.index)
assert (hvg_positions>=0).all()
assert np.array_equal(hvg_var.index,hvg_final_var.index)
hvg_annotation_rank_differs=not np.allclose(hvg_var.highly_variable_rank,hvg_final_var.highly_variable_rank,equal_nan=True)
# 06 recomputed HVG after 05 annotation/subsetting; only the final 06 rank is authoritative.
hvg_var=hvg_final_var.copy()
print("Using frozen 06 HVG ranking; older annotation rank differs:",hvg_annotation_rank_differs,flush=True)
hvg_obs.to_csv("../results/sensitivity/current_cohort_20261030/PCA_source_cell_metadata.tsv",sep="\t")
# PCA uses the same sklearn PCA/ARPACK backend called by installed Scanpy zero_center=True.
# The cached seurat_v3 ranking is frozen in 06, computed without age labels.
# With no batch key this ranking defines the nested top-2000/3000/5000 sets exactly.
with h5py.File("../data_interim/cell_type_annotated.h5ad","r") as handle:
    hvg_log_matrix=read_elem(handle["X"])
print("Loaded frozen log-normalised sparse matrix once:",hvg_log_matrix.shape,flush=True)
hvg_audit=[];hvg_cache={}
for n_hvg in [2000,3000,5000]:
    started=time.monotonic()
    keep=hvg_var.highly_variable_rank.lt(n_hvg).to_numpy()
    assert keep.sum()==n_hvg
    matrix=hvg_log_matrix[:,np.flatnonzero(keep)].tocsr()
    with threadpool_limits(limits=2):
        model=PCA(n_components=50,svd_solver="arpack",random_state=0)
        coords=model.fit_transform(matrix).astype(np.float32)
    assert coords.shape==(len(hvg_obs),50) and np.isfinite(coords).all()
    np.save(f"../results/sensitivity/current_cohort_20261030/PCA_HVG{n_hvg}.npy",coords[hvg_positions])
    pd.DataFrame({"gene":hvg_var.index[keep],"frozen_rank":hvg_var.highly_variable_rank.loc[keep]}).to_csv(f"../results/sensitivity/current_cohort_20261030/genes_HVG{n_hvg}.tsv",sep="\t",index=False)
    hvg_audit.append(dict(HVG=n_hvg,n_fit_cells=len(hvg_obs),n_target_cells=len(hvg_positions),n_PCs=50,solver="arpack",random_state=0,seconds=time.monotonic()-started,scope="same all-annotated-cell fit population as frozen 06; age-blind cached seurat_v3 ranking"))
    hvg_cache[n_hvg]=coords[hvg_positions]
    if n_hvg==5000:
        alignment=np.sum(hvg_cache[n_hvg]*hvg_frozen_pca,axis=0)
        signs=np.where(alignment>=0,1.,-1.)
        error=float(np.max(abs(hvg_cache[n_hvg]*signs-hvg_frozen_pca)))
        print("HVG5000 anchor max coordinate error after sign alignment:",error,flush=True)
        pd.DataFrame([dict(max_coordinate_error_after_sign_alignment=error,same_fit_population=True,same_frozen_HVG_ranking=True)]).to_csv("../results/sensitivity/current_cohort_20261030/HVG5000_anchor_check.tsv",sep="\t",index=False)
    pd.DataFrame(hvg_audit).to_csv("../results/sensitivity/current_cohort_20261030/HVG_PCA_run_log.tsv",sep="\t",index=False)
    print("HVG",n_hvg,"PCA complete; seconds",round(hvg_audit[-1]["seconds"],1),flush=True)
    del matrix,coords,model;gc.collect()


Using frozen 06 HVG ranking; older annotation rank differs: True
Loaded frozen log-normalised sparse matrix once: (118105, 20138)
HVG 2000 PCA complete; seconds 3.8
HVG 3000 PCA complete; seconds 5.3
HVG5000 anchor max coordinate error after sign alignment: 0.0005483627319335938
HVG 5000 PCA complete; seconds 8.5


## 2. 参数全组合与条件敏感性

HVG×PC×细胞门槛×聚合方式共72组合，另加冻结基、QC、抽样、sex和复合mouse.id控制；不可估计设置和稀疏年龄组逐条保存。age-adjusted/sex-adjusted不混作同一种轴定义。复合mouse.id是否实际池化不作无依据假定，提供剔除与可能重叠块重抽样对照。


In [3]:
# Current 06/08 inputs only. Main-cell threshold comes from the user-confirmed week-10 parameters.
params=json.load(open("../results/aging_axis/current_cohort_20261030/parameters.json",encoding="utf-8"))
MAIN_MIN_CELLS=int(params["min_cells"]);MAIN_N_PC=int(params["n_pcs"])
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    sens_obs=read_elem(handle["obs"]).rename(columns={"mouse.id":"mouse_id"})
    sens_coords=np.asarray(read_elem(handle["obsm/X_pca"]))
for key in ["mouse_id","tissue","mct","age_group","sex"]:sens_obs[key]=sens_obs[key].astype(str)
sens_obs["age"]=pd.to_numeric(sens_obs.age)
sens_obs=sens_obs.reset_index(drop=True)
mean_table=pd.read_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv",sep="\t")
trim_table=pd.read_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_trimmed.tsv",sep="\t")
cohort_keys=set(mean_table[["mouse_id","tissue","mct"]].itertuples(index=False,name=None))
reference=evaluate_axes(mean_table.loc[mean_table.n_cells.ge(MAIN_MIN_CELLS)],MAIN_N_PC)
assert reference["eligible"]==params["primary_nodes"]
sens_rows=[];sens_pairs=[];sens_failures=[];sens_configs=[];sens_lomo=[]

def run_scenario(name,table,k=7,model="centroid",family="other",**settings):
    trial=evaluate_axes(table,k,model)
    compar=compare_scenarios(reference,trial)
    sens_configs.append(dict(setting=name,family=family,n_pcs=k,n_mouse_nodes=len(table),n_nodes_formal=len(trial["eligible"]),**compar,**settings))
    for record in trial["summary"].to_dict("records"):
        node=record["node"]
        base=reference["lomo"].loc[reference["lomo"].node.eq(node),["mouse_id","score"]]
        held=trial["lomo"].loc[trial["lomo"].node.eq(node),["mouse_id","score"]] if len(trial["lomo"]) else pd.DataFrame(columns=["mouse_id","score"])
        joined=base.merge(held,on="mouse_id",suffixes=("_base","_trial")).dropna()
        record.update(setting=name,family=family,n_pcs=k,n_score_pairs=len(joined),crossfit_score_spearman=describe_age(joined.score_trial,joined.score_base) if len(joined) else np.nan,**settings)
        sens_rows.append(record)
        if record["status"]!="complete":
            sens_failures.append(dict(setting=name,family=family,node=node,status=record["status"],reason=record.get("reason","") or "fewer_than_two_young_or_old_mice",n_young=record["n_young"],n_old=record["n_old"]))
    if len(trial["lomo"]):
        fold=trial["lomo"].copy();fold["setting"]=name;fold["family"]=family;sens_lomo.append(fold)
    for i,a in enumerate(trial["eligible"]):
        for b in trial["eligible"][i+1:]:
            sens_pairs.append(dict(setting=name,family=family,node1=a,node2=b,cosine=trial["cosine"].loc[a,b],is_primary_pair=a in reference["eligible"] and b in reference["eligible"]))
    return trial

# All 72 configurations are retained, not only favourable results.
for hvg in [2000,3000,5000]:
    coords=np.load(f"../results/sensitivity/current_cohort_20261030/PCA_HVG{hvg}.npy")
    for aggregation in ["mean","trimmed"]:
        table=cohort_filter(aggregate_coordinates(sens_obs,coords,aggregation),cohort_keys,0)
        for minimum in [0,30,50]:
            selected=table.loc[table.n_cells.ge(minimum)]
            for k in [7,10,20,30]:
                run_scenario(f"HVG{hvg}_PC{k}_min{minimum}_{aggregation}",selected,k,family="factorial",HVG=hvg,min_cells=minimum,aggregation=aggregation)
    print("Completed full factorial HVG",hvg,flush=True)
# Single-factor PC and aggregation controls on the exact frozen 06 basis.
for k in [7,10,20,30]:
    for aggregation,table in [("mean",mean_table),("trimmed",trim_table)]:
        for minimum in [0,30,50]:
            run_scenario(f"frozen_PC{k}_min{minimum}_{aggregation}",table.loc[table.n_cells.ge(minimum)],k,family="frozen_basis",HVG=5000,min_cells=minimum,aggregation=aggregation)
# Stricter QC can be applied to frozen annotated cells. Relaxing QC cannot restore unannotated failed cells.
qc_limits={"Kidney":(25000,6000),"Liver":(45000,6000),"Lung":(16000,4000),"spleen/marrow":(30000,5500)}
qc_audit=[]
for label,min_genes,min_counts,upper in [("stricter_1",300,300,.9),("stricter_2",500,500,.8)]:
    cellmask=np.zeros(len(sens_obs),bool)
    for tissue,(maxcounts,maxgenes) in qc_limits.items():
        select=sens_obs.tissue.eq(tissue)&sens_obs.n_genes_by_counts.ge(min_genes)&sens_obs.total_counts.ge(min_counts)&sens_obs.total_counts.le(maxcounts*upper)&sens_obs.n_genes_by_counts.le(maxgenes*upper)
        cellmask|=select.to_numpy()
        qc_audit.append(dict(setting=label,tissue=tissue,min_genes=min_genes,min_counts=min_counts,max_genes=maxgenes*upper,max_counts=maxcounts*upper,n_cells_retained=int(select.sum()),scope="stricter cell selection on frozen annotated singlets and fixed PCA basis"))
    table=cohort_filter(aggregate_coordinates(sens_obs,sens_coords,cell_mask=cellmask),cohort_keys,MAIN_MIN_CELLS)
    run_scenario("QC_"+label,table,MAIN_N_PC,family="QC_stricter",HVG=5000,min_cells=MAIN_MIN_CELLS,aggregation="mean")
sens_failures.append(dict(setting="QC_relaxed",family="QC",node="ALL",status="not_estimable",reason="Cells rejected upstream lack current lineage annotations; no invented labels or silent restoration. Only stricter QC evaluated."))
# Cell resampling measures coordinate precision; mouse IDs remain the statistical units.
for cap in [100,300]:
    for seed in [2026,2027,2028]:
        table=cohort_filter(aggregate_coordinates(sens_obs,sens_coords,cap=cap,seed=seed),cohort_keys,MAIN_MIN_CELLS)
        run_scenario(f"cell_cap{cap}_seed{seed}",table,MAIN_N_PC,family="cell_subsampling",HVG=5000,min_cells=MAIN_MIN_CELLS,aggregation="mean",cap=cap,seed=seed)
primary=mean_table.loc[mean_table.n_cells.ge(MAIN_MIN_CELLS)]
run_scenario("sex_adjusted",primary,MAIN_N_PC,model="sex_adjusted",family="sex",HVG=5000,min_cells=MAIN_MIN_CELLS,aggregation="mean")
run_scenario("male_only",primary.loc[primary.sex.eq("male")],MAIN_N_PC,family="sex",HVG=5000,min_cells=MAIN_MIN_CELLS,aggregation="mean")
run_scenario("drop_composite_mouse_ids",primary.loc[~primary.mouse_id.str.contains("/",regex=False)],MAIN_N_PC,family="identity",HVG=5000,min_cells=MAIN_MIN_CELLS,aggregation="mean")
pd.DataFrame(qc_audit).to_csv("../results/sensitivity/current_cohort_20261030/QC_threshold_audit.tsv",sep="\t",index=False)
pd.DataFrame(sens_configs).to_csv("../results/sensitivity/current_cohort_20261030/scenario_summary.tsv",sep="\t",index=False)
pd.DataFrame(sens_rows).to_csv("../results/sensitivity/current_cohort_20261030/sensitivity_summary.tsv",sep="\t",index=False)
pd.DataFrame(sens_rows).to_csv("../results/sensitivity/sensitivity_summary.tsv",sep="\t",index=False)
pd.DataFrame(sens_pairs).to_csv("../results/sensitivity/current_cohort_20261030/scenario_pairwise_cosines.tsv",sep="\t",index=False)
pd.concat(sens_lomo,ignore_index=True).to_csv("../results/sensitivity/current_cohort_20261030/all_scenario_held_out_scores.tsv",sep="\t",index=False)
pd.DataFrame(sens_failures).to_csv("../results/sensitivity/current_cohort_20261030/failures.tsv",sep="\t",index=False)
print("All",len(sens_configs),"scenarios and",len(sens_rows),"node-setting records saved; failures",len(sens_failures),flush=True)


Completed full factorial HVG 2000
Completed full factorial HVG 3000
Completed full factorial HVG 5000
All 107 scenarios and 856 node-setting records saved; failures 237


## 3. 鼠级验证与bootstrap B=500

全局按年龄组重抽mouse.id，同一编号跨组织共享权重；组缺失不补零。潜在3-M-7/8与3-M-8重叠另作共同块控制，不宣称已核实物理动物身份。


In [4]:
# Full-data direction and held-mouse projections from week 10 are descriptive.
# Bootstrap resamples mouse IDs jointly across tissues within age strata.
primary=mean_table.loc[mean_table.n_cells.ge(MAIN_MIN_CELLS)].copy()
bootdir,bootpair=full_bootstrap(primary,MAIN_N_PC,B=500,seed=SEED)
bootdir.to_csv("../results/sensitivity/current_cohort_20261030/bootstrap_B500_directions.tsv",sep="\t",index=False)
bootpair.to_csv("../results/sensitivity/current_cohort_20261030/bootstrap_B500_pairs.tsv",sep="\t",index=False)
boot_failures=bootdir.loc[bootdir.status.eq("not_estimable")].copy()
boot_failures.to_csv("../results/sensitivity/current_cohort_20261030/bootstrap_failures.tsv",sep="\t",index=False)
direction_ci=[]
for node,g in bootdir.groupby("node",sort=True):
    vals=g.cosine_to_baseline.dropna();norms=g.direction_norm.dropna()
    direction_ci.append(dict(node=node,n_bootstrap_valid=len(vals),valid_fraction=len(vals)/500,cosine_CI_low=vals.quantile(.025),cosine_CI_high=vals.quantile(.975),norm_CI_low=norms.quantile(.025),norm_CI_high=norms.quantile(.975)))
direction_ci=pd.DataFrame(direction_ci)
direction_ci.to_csv("../results/sensitivity/current_cohort_20261030/bootstrap_direction_CI.tsv",sep="\t",index=False)
pair_ci=[]
for (a,b),g in bootpair.groupby(["node1","node2"],sort=True):
    v=g.cosine.dropna()
    pair_ci.append(dict(node1=a,node2=b,cosine=reference["cosine"].loc[a,b],n_bootstrap_valid=len(v),CI_low=v.quantile(.025),CI_high=v.quantile(.975)))
pair_ci=pd.DataFrame(pair_ci)
pair_ci.to_csv("../results/aging_axis/current_cohort_20261030/pairwise_cosine_CI.tsv",sep="\t",index=False)
pair_ci.to_csv("../results/sensitivity/current_cohort_20261030/pairwise_cosine_CI.tsv",sep="\t",index=False)
# Global LOMO drops the same mouse from every tissue before recomputing all directions.
global_lomo=[]
for mouse in sorted(primary.mouse_id.unique()):
    train=primary.loc[primary.mouse_id.ne(mouse)]
    fits={node:fit_axis(train.loc[train.node.eq(node)],MAIN_N_PC) for node in reference["eligible"]}
    for i,a in enumerate(reference["eligible"]):
        for b in reference["eligible"][i+1:]:
            value=axis_cos(fits[a]["d"],fits[b]["d"]) if "d" in fits[a] and "d" in fits[b] else np.nan
            global_lomo.append(dict(mouse_left_out=mouse,node1=a,node2=b,cosine=value,baseline_cosine=reference["cosine"].loc[a,b],status="complete" if fits[a]["formal"] and fits[b]["formal"] else "exploratory_or_not_estimable"))
pd.DataFrame(global_lomo).to_csv("../results/sensitivity/current_cohort_20261030/global_lomo_pairwise.tsv",sep="\t",index=False)
# Composite IDs are not asserted to be individual mice. A conservative shared-ID block control is saved.
blocked=primary.copy()
blocked["mouse_id"]=blocked.mouse_id.replace({"3-M-7/8":"3-M-7_8_overlap_block","3-M-8":"3-M-7_8_overlap_block"})
assert not blocked.duplicated(["node","mouse_id"]).any()
blockdir,blockpair=full_bootstrap(blocked,MAIN_N_PC,B=500,seed=SEED)
blockdir.to_csv("../results/sensitivity/current_cohort_20261030/composite_block_bootstrap_directions.tsv",sep="\t",index=False)
blockpair.to_csv("../results/sensitivity/current_cohort_20261030/composite_block_bootstrap_pairs.tsv",sep="\t",index=False)
pd.DataFrame([dict(original_mouse_ids=primary.mouse_id.nunique(),conservative_overlap_blocks=blocked.mouse_id.nunique(),composite_ids="3-M-5/6;3-M-7/8",policy="No pooling interpretation asserted as fact. Exclude slash IDs and co-resample possible 7/8 versus 8 overlap as sensitivity controls.")]).to_csv("../results/sensitivity/current_cohort_20261030/mouse_identity_audit.tsv",sep="\t",index=False)
print("Completed age-stratified paired-mouse bootstrap B=500 plus conservative overlap-block control.",flush=True)


Completed age-stratified paired-mouse bootstrap B=500 plus conservative overlap-block control.


## 4. 稳定性评级与阈值扫描

连续指标优先；阈值是明确披露的操作性标准，并非预注册或显著性检验。对稀疏、不稳定及缺乏单调年龄梯度的结果明确降级。不能把PCA稳定性解释为09全部通路已获独立验证。


In [5]:
# Operational stability criteria are reported explicitly, not represented as preregistered.
sensitivity=pd.read_csv("../results/sensitivity/current_cohort_20261030/sensitivity_summary.tsv",sep="\t")
configs=pd.read_csv("../results/sensitivity/current_cohort_20261030/scenario_summary.tsv",sep="\t")
baseline=pd.read_csv("../results/aging_axis/current_cohort_20261030/node_summary.tsv",sep="\t").set_index("node")
direction_ci=pd.read_csv("../results/sensitivity/current_cohort_20261030/bootstrap_direction_CI.tsv",sep="\t")
pair_ci=pd.read_csv("../results/sensitivity/current_cohort_20261030/pairwise_cosine_CI.tsv",sep="\t")
factorial=sensitivity.loc[sensitivity.family.eq("factorial")]
assert configs.setting.nunique()==107 and len(sensitivity)==856
assert factorial.setting.nunique()==72
verdicts=[]
for node in NODES:
    g=factorial.loc[factorial.node.eq(node)]
    formal=g.status.eq("complete")
    boot=direction_ci.loc[direction_ci.node.eq(node)]
    ci=float(boot.cosine_CI_low.iloc[0]) if len(boot) else np.nan
    coverage=float(boot.valid_fraction.iloc[0]) if len(boot) else 0.
    estimable=float(formal.mean())
    score_consistency=float((formal&g.crossfit_score_spearman.ge(.7)).mean())
    age_consistency=float((formal&g.rho_age_crossfit.ge(.3)).mean())
    base=baseline.loc[node];reasons=[]
    if not bool(base.formal):reasons.append("baseline_low_endpoint_replication")
    if not np.isfinite(base.get("min_lomo_cosine",np.nan)) or base.get("min_lomo_cosine",np.nan)<.7:reasons.append("single_mouse_sensitive")
    if not np.isfinite(ci) or ci<.5:reasons.append("bootstrap_direction_CI_low")
    if coverage<.9:reasons.append("bootstrap_missing_groups")
    if estimable<.8:reasons.append("insufficient_coverage_across_settings")
    if score_consistency<.8:reasons.append("crossfit_scores_setting_sensitive")
    stable=not reasons
    age_supported=stable and base.get("rho_age_crossfit",np.nan)>=.3 and age_consistency>=.8
    verdicts.append(dict(node=node,n_young=int(base.n_young),n_old=int(base.n_old),baseline_formal=bool(base.formal),rho_age_crossfit=base.get("rho_age_crossfit",np.nan),min_lomo_cosine=base.get("min_lomo_cosine",np.nan),bootstrap_cosine_CI_low=ci,bootstrap_valid_fraction=coverage,n_factorial_settings=len(g),formal_setting_fraction=estimable,score_consistent_fraction=score_consistency,positive_age_trend_fraction=age_consistency,stable_direction=stable,consistent_monotone_age_trend=bool(age_supported),interpretation="stable_direction" if stable else "downgraded_exploratory_or_setting_sensitive",reasons=";".join(reasons)))
node_verdict=pd.DataFrame(verdicts)
node_verdict["stable_direction_conditional_on_primary_definition"]=node_verdict.stable_direction
extra=sensitivity.loc[sensitivity.family.isin(["QC_stricter","cell_subsampling","sex","identity"])]
node_verdict["extra_control_consistent_fraction"]=node_verdict.node.map(extra.assign(pass_control=lambda d:d.status.eq("complete")&d.crossfit_score_spearman.ge(.7)).groupby("node").pass_control.mean())
sex_rows=extra.loc[extra.setting.eq("sex_adjusted")].set_index("node")
node_verdict["sex_adjusted_rho_age"]=node_verdict.node.map(sex_rows.rho_age_crossfit)
node_verdict["sex_adjusted_score_correlation"]=node_verdict.node.map(sex_rows.crossfit_score_spearman)
node_verdict["sex_adjustment_sensitive"]=node_verdict.sex_adjusted_score_correlation.lt(.7)|node_verdict.sex_adjusted_score_correlation.isna()
node_verdict["overall_robust"]=node_verdict.stable_direction & node_verdict.extra_control_consistent_fraction.ge(.8)&~node_verdict.sex_adjustment_sensitive
node_verdict["consistent_monotone_age_trend"]=node_verdict.consistent_monotone_age_trend&node_verdict.overall_robust&node_verdict.sex_adjusted_rho_age.ge(.3)
node_verdict.loc[node_verdict.stable_direction&~node_verdict.overall_robust,"interpretation"]="conditional_main_direction_stability_but_control_sensitive"
node_verdict.loc[node_verdict.sex_adjustment_sensitive,"reasons"]=node_verdict.loc[node_verdict.sex_adjustment_sensitive,"reasons"].fillna("")+";sex_adjustment_sensitive"
node_verdict.to_csv("../results/sensitivity/current_cohort_20261030/node_stability_verdict.tsv",sep="\t",index=False)
pair_scenarios=pd.read_csv("../results/sensitivity/current_cohort_20261030/scenario_pairwise_cosines.tsv",sep="\t")
pair_checks=[]
for record in pair_ci.to_dict("records"):
    a,b=record["node1"],record["node2"]
    g=pair_scenarios.loc[pair_scenarios.family.eq("factorial")&pair_scenarios.node1.eq(a)&pair_scenarios.node2.eq(b)]
    agreement=float((np.sign(g.cosine)==np.sign(record["cosine"])).sum()/72)
    node_ok=node_verdict.set_index("node").loc[[a,b],"overall_robust"].all()
    ci_excludes_zero=record["CI_low"]>0 or record["CI_high"]<0
    record.update(formal_setting_fraction=len(g)/72,sign_agreement_fraction=agreement,both_nodes_stable=bool(node_ok),consistent_descriptive_similarity=bool(ci_excludes_zero and agreement>=.8 and len(g)/72>=.8 and node_ok),note="Bootstrap interval and consistency are descriptive; no multiplicity-adjusted pairwise significance or causality claimed.")
    pair_checks.append(record)
pair_verdict=pd.DataFrame(pair_checks)
pair_verdict.to_csv("../results/sensitivity/current_cohort_20261030/pair_similarity_stability.tsv",sep="\t",index=False)
# Threshold scans expose dependence on the operational grading rules; do not select an optimum.
scan=[]
for metric,values in [("bootstrap_CI_low",[.3,.4,.5,.6,.7,.8,.9]),("min_lomo_cosine",[.5,.6,.7,.8,.9]),("score_consistent_fraction",[.6,.7,.8,.9])]:
    for value in values:
        ci_floor=value if metric=="bootstrap_CI_low" else .5
        lo_floor=value if metric=="min_lomo_cosine" else .7
        sc_floor=value if metric=="score_consistent_fraction" else .8
        passed=node_verdict.baseline_formal&node_verdict.bootstrap_cosine_CI_low.ge(ci_floor)&node_verdict.min_lomo_cosine.ge(lo_floor)&node_verdict.bootstrap_valid_fraction.ge(.9)&node_verdict.formal_setting_fraction.ge(.8)&node_verdict.score_consistent_fraction.ge(sc_floor)
        scan.append(dict(metric=metric,threshold=value,n_stable_unadjusted_primary=int(passed.sum()),stable_nodes=";".join(node_verdict.loc[passed,"node"])))
pd.DataFrame(scan).to_csv("../results/sensitivity/current_cohort_20261030/stability_threshold_scan.tsv",sep="\t",index=False)
print(node_verdict[["node","bootstrap_cosine_CI_low","formal_setting_fraction","score_consistent_fraction","stable_direction","consistent_monotone_age_trend","reasons"]].to_string(index=False))
print("Consistent descriptive pairs:",int(pair_verdict.consistent_descriptive_similarity.sum()),"/",len(pair_verdict))


                    node  bootstrap_cosine_CI_low  formal_setting_fraction  score_consistent_fraction  stable_direction  consistent_monotone_age_trend                                                                                                                                                       reasons
       Kidney | lymphoid                 0.005228                 0.333333                   0.291667             False                          False                                   bootstrap_direction_CI_low;insufficient_coverage_across_settings;crossfit_scores_setting_sensitive;sex_adjustment_sensitive
        Kidney | myeloid                 0.697996                 1.000000                   1.000000              True                          False                                                                                                                                     ;sex_adjustment_sensitive
        Liver | lymphoid                      NaN                 0.00000

## 5. 完整敏感性图

包括全部节点、全部72组合、全部21主图节点对及额外控制，不仅展示有利结果。灰色表示不可正式估计，不补零。


In [6]:
from matplotlib.colors import ListedColormap
boot_direction=pd.read_csv("../results/sensitivity/current_cohort_20261030/bootstrap_B500_directions.tsv",sep="\t")
with PdfPages("../figures/supplement/stability_plots.pdf") as pdf:
    # 1. Direction uncertainty and out-of-fold age trends, including every node.
    fig,axes=plt.subplots(1,2,figsize=(16,7),layout="constrained")
    names=NODES;ordered=node_verdict.set_index("node").reindex(names)
    for i,node in enumerate(names):
        row=direction_ci.loc[direction_ci.node.eq(node)]
        if len(row):
            mid=float(boot_direction.loc[boot_direction.node.eq(node),"cosine_to_baseline"].median())
            low=float(row.cosine_CI_low.iloc[0]);high=float(row.cosine_CI_high.iloc[0])
            color="#2166ac" if bool(ordered.loc[node,"overall_robust"]) else ("#e6ab02" if bool(ordered.loc[node,"stable_direction"]) else "#b2182b")
            axes[0].plot([low,high],[i,i],color=color,lw=2);axes[0].plot(mid,i,"o",color=color)
        else:axes[0].text(0,i,"exploratory / no primary CI",va="center",fontsize=8)
        rho=ordered.loc[node,"rho_age_crossfit"]
        if np.isfinite(rho):axes[1].barh(i,rho,color="#315b72")
    for ax in axes:
        ax.set_yticks(range(len(names)),[n.replace(" | ","\n") for n in names],fontsize=8);ax.invert_yaxis();ax.axvline(0,color="#777777",lw=.8)
    axes[0].axvline(.5,color="#999999",ls="--");axes[0].set(xlim=(-1,1),xlabel="Bootstrap cosine to full direction: median and 95% interval",title="B=500, age-stratified shared-mouse resampling")
    axes[1].set(xlim=(-1,1),xlabel="Descriptive Spearman rho",title="Held-mouse score versus age; no naive P value")
    fig.suptitle("Direction stability and cross-fitted age trend")
    pdf.savefig(fig);fig.savefig("../figures/supplement/stability_plots.png",dpi=150);plt.close(fig)
    # 2. All factorial settings; missing/low-replication settings are grey, not omitted.
    subset=sensitivity.loc[sensitivity.family.eq("factorial")].copy()
    setting_order=configs.loc[configs.family.eq("factorial"),"setting"].tolist()
    vals=subset.pivot(index="node",columns="setting",values="crossfit_score_spearman").reindex(index=names,columns=setting_order)
    statuses=subset.pivot(index="node",columns="setting",values="status").reindex(index=names,columns=setting_order)
    colors=vals.where(statuses.eq("complete"))
    cmap=plt.get_cmap("RdBu_r").copy();cmap.set_bad("#dddddd")
    fig,ax=plt.subplots(figsize=(18,7))
    im=ax.imshow(colors,cmap=cmap,vmin=-1,vmax=1,aspect="auto")
    labels=[]
    for name in setting_order:
        r=configs.loc[configs.setting.eq(name)].iloc[0]
        labels.append(f"P{int(r.n_pcs)} N{int(r.min_cells) if int(r.min_cells)>0 else 'base'} "+("M" if r.aggregation=="mean" else "T"))
    ax.set_xticks(range(72),labels,rotation=90,fontsize=6);ax.set_yticks(range(8),names,fontsize=8)
    for x in [23.5,47.5]:ax.axvline(x,color="black",lw=2)
    ax.set_title("All 72 settings: HVG2000 | HVG3000 | HVG5000; P=PCs; Nbase=current 08 cohort, N30/50=min cells; M=mean, T=trimmed")
    fig.colorbar(im,ax=ax,label="Held-mouse score rank correlation with frozen primary")
    fig.tight_layout();pdf.savefig(fig);plt.close(fig)
    # 3. All primary pair intervals, not only strongest similarities.
    p=pair_verdict.copy().sort_values("cosine")
    fig,ax=plt.subplots(figsize=(12,10))
    for i,r in enumerate(p.itertuples()):
        ax.plot([r.CI_low,r.CI_high],[i,i],color="#2166ac" if r.consistent_descriptive_similarity else "#999999",lw=1.8)
        ax.plot(r.cosine,i,"o",color="#315b72")
    ax.set_yticks(range(len(p)),[a+" / "+b for a,b in zip(p.node1,p.node2)],fontsize=7)
    ax.axvline(0,color="black",ls="--",lw=1);ax.set(xlim=(-1,1),xlabel="Original cosine and mouse-bootstrap 95% interval",title="All 21 primary node pairs; descriptive intervals, not FDR significance")
    fig.tight_layout();pdf.savefig(fig);plt.close(fig)
    # 4. QC, subsampling, sex and ambiguous-ID controls.
    controls=sensitivity.loc[sensitivity.family.isin(["QC_stricter","cell_subsampling","sex","identity"])]
    settings=configs.loc[configs.family.isin(["QC_stricter","cell_subsampling","sex","identity"]),"setting"].tolist()
    values=controls.pivot(index="node",columns="setting",values="crossfit_score_spearman").reindex(index=NODES,columns=settings)
    stat=controls.pivot(index="node",columns="setting",values="status").reindex(index=NODES,columns=settings)
    fig,ax=plt.subplots(figsize=(15,7));im=ax.imshow(values.where(stat.eq("complete")),cmap=cmap,vmin=-1,vmax=1,aspect="auto")
    ax.set_xticks(range(len(settings)),settings,rotation=45,ha="right",fontsize=8);ax.set_yticks(range(8),NODES,fontsize=8)
    ax.set_title("Additional controls: frozen PCA basis; grey = not formally estimable")
    fig.colorbar(im,ax=ax,label="Held-mouse score rank correlation");fig.tight_layout();pdf.savefig(fig);plt.close(fig)
print("Saved complete 4-page sensitivity figures.")


Saved complete 4-page sensitivity figures.


## 6. 不稳定方向的备用结果与运行问题记录

按任务书保留样本级距离，不从弱或敏感的向量强行推断共同衰老方向；鼠对数不是独立生物学重复数。


In [7]:
# Task-book fallback: sample-level distances rather than asserting a stable shared direction.
from scipy.spatial.distance import pdist, squareform
fallback_mean=pd.read_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv",sep="\t")
cols=[f"PC{i+1}" for i in range(7)]
distance_rows=[]
for node,g in fallback_mean.groupby("node",sort=True):
    distances=squareform(pdist(g[cols].to_numpy(float),"euclidean"))
    records=g.reset_index(drop=True)
    for i in range(len(records)):
        for j in range(i+1,len(records)):
            a,b=records.iloc[i],records.iloc[j]
            ages=sorted([a.age_group,b.age_group])
            distance_rows.append(dict(node=node,mouse1=a.mouse_id,mouse2=b.mouse_id,age_pair=" / ".join(ages),distance=distances[i,j],sex_pair=" / ".join(sorted([a.sex,b.sex]))))
distance_rows=pd.DataFrame(distance_rows)
distance_rows.to_csv("../results/aging_axis/current_cohort_20261030/within_node_sample_distances.tsv",sep="\t",index=False)
distance_summary=distance_rows.groupby(["node","age_pair"],as_index=False).agg(n_pairs=("distance","size"),mean_distance=("distance","mean"),median_distance=("distance","median"))
distance_summary["note"]="Mouse-pair distances are dependent; n_pairs is not the number of biological replicates. Descriptive only."
distance_summary.to_csv("../results/aging_axis/current_cohort_20261030/sample_distance_summary.tsv",sep="\t",index=False)
pd.DataFrame([
 {"stage":"representation_preflight","issue":"Broad Scanpy import overhead; interrupted before PCA output","resolution":"Use the identical sklearn PCA/ARPACK backend called by installed Scanpy; record runtime and anchor reproduction."},
 {"stage":"representation_preflight","issue":"Older annotated-object HVG ranking differs from recomputed frozen 06 ranking","resolution":"Use authoritative final_data.var highly_variable_rank; exact gene order matched; frozen 06 PCA anchor checked."}
]).to_csv("../results/sensitivity/current_cohort_20261030/preflight_resolution_log.tsv",sep="\t",index=False)
print("Saved task-book sample-distance fallback without treating cell pairs as independent repeats.")


Saved task-book sample-distance fallback without treating cell pairs as independent repeats.


## 7. 任务书逐项交付与解释边界

保存完整报告、当前参数和不稳定结论；不把分析完成与生物学假设获得支持混为一谈。


In [8]:
import json
import hashlib
import importlib.metadata
import pandas as pd
import numpy as np
params=json.load(open("../results/aging_axis/current_cohort_20261030/parameters.json",encoding="utf-8"))
node_verdict=pd.read_csv("../results/sensitivity/current_cohort_20261030/node_stability_verdict.tsv",sep="\t")
pair_verdict=pd.read_csv("../results/sensitivity/current_cohort_20261030/pair_similarity_stability.tsv",sep="\t")
configurations=pd.read_csv("../results/sensitivity/current_cohort_20261030/scenario_summary.tsv",sep="\t")
failures=pd.read_csv("../results/sensitivity/current_cohort_20261030/failures.tsv",sep="\t")
coverage=pd.read_csv("../results/aging_axis/current_cohort_20261030/node_coverage_by_min_cells.tsv",sep="\t")
anchor=pd.read_csv("../results/sensitivity/current_cohort_20261030/HVG5000_anchor_check.tsv",sep="\t")
# Identity of the main representation is checked with basis-invariant pair similarities too.
pairs=pd.read_csv("../results/sensitivity/current_cohort_20261030/scenario_pairwise_cosines.tsv",sep="\t")
a=pairs.loc[pairs.setting.eq("HVG5000_PC7_min0_mean")]
b=pairs.loc[pairs.setting.eq("frozen_PC7_min0_mean")]
matched=a.merge(b,on=["node1","node2"],suffixes=("_recomputed","_frozen"),validate="one_to_one")
anchor_pair_error=float(abs(matched.cosine_recomputed-matched.cosine_frozen).max())
assert anchor_pair_error<1e-3
pd.DataFrame([dict(n_pairs=len(matched),max_pair_cosine_error=anchor_pair_error)]).to_csv("../results/sensitivity/current_cohort_20261030/PCA_basis_invariant_anchor_check.tsv",sep="\t",index=False)
summary=node_verdict[["node","rho_age_crossfit","min_lomo_cosine","bootstrap_cosine_CI_low","formal_setting_fraction","score_consistent_fraction","sex_adjusted_rho_age","overall_robust"]]
# Simple Markdown generation without optional table packages.
headers=summary.columns.tolist()
table="| "+" | ".join(headers)+" |\n|"+ "|".join(["---"]*len(headers))+"|\n"
for row in summary.itertuples(index=False,name=None):
    table+="| "+" | ".join(f"{v:.3f}" if isinstance(v,(float,np.floating)) and np.isfinite(v) else str(v).replace("|"," / ") for v in row)+" |\n"
week10_notes="""# 第十周：PCA衰老方向与组织相似性（当前队列，2026-10-30）

## 已完成与当前口径
用户确认保留08原117个mouse×tissue×mct样本作为主分析；涵盖22个mouse.id统计单元。主表示为06冻结的细胞级X_pca按鼠×节点求普通均值，PC1–7（HVG=5000、log-normalised、未scale），不使用旧Heart/内皮/巨噬细胞/Harmony数据。
年龄标签沿用06：1/3月young，18/21/24月middle，30月old；不按旧YAML把24月改为old。四组织为Kidney、Liver、Lung、spleen/marrow，两类为lymphoid/myeloid。

方向d=old中心−young中心；每个mouse.id权重相同。middle不参与centroid方向训练，仅投影并用于描述连续年龄趋势。
正式相似性热图保留young/old各≥2个mouse.id的7节点；Liver×lymphoid仅1个young，继续探索性并从正式热图排除。
pca_scores.csv保留全部117行，包含训练内投影及留一鼠投影；删除唯一young后的投影不可估计，不补0。

## 验证和解释
小型解析模拟检查方向、符号、年轻中心零点、老组投影等于方向长度；实际数据验证节点×mouse.id唯一、同鼠跨组织年龄/sex一致及余弦对称对角1。
留出mouse.id不参与young中心及方向估计；PCA/HVG属于06冻结表示，不在各折重训，因此只能称固定表示条件下的内部验证，不是完全嵌套或外部验证。
训练内young/old分离由方向定义产生，不据此计算年龄P值；留鼠Spearman相关仅描述，不给存在训练折依赖的得分套用朴素P值或BH。

全部old为雄性。sex加性对照在每训练折拟合PC~sex+age_group，投影留出样本时移除该训练折估计的female系数，以male为参考；没有使用留出样本拟合系数。额外计算sex加性调整与仅雄鼠对照，主图本身遵循任务书的未校正中心差定义，不据此推断雌鼠衰老或因果关系。
存在3-M-5/6、3-M-7/8复合编号；不无依据断言其物理动物数。第十一周提供剔除复合编号和可能7/8与8重叠的保守共同块bootstrap。

## 本轮结论
原始相似性只是描述性几何结果。第十一周显示只有肾髓系未校正主方向通过基础方向稳定性条件，但sex调整后的年龄得分趋势明显减弱；21个主图节点对均未通过联合稳健性标准。图中数值较高不能直接写成稳定的跨组织共同衰老方向。
详见docs/result_stability_notes.md。按任务书风险预案另外输出各节点鼠级距离及年龄组距离摘要，不把鼠对当独立重复。

## 文件
- notebooks/10_aging_axis.ipynb：全部Python逻辑内嵌、相对地址字面量。
- src/aging_axis.py：同步当前实现并保留历史API；Notebook不从外部导入分析逻辑。
- results/aging_axis/pca_scores.csv、cosine_similarity.csv：任务书指定主入口。
- results/aging_axis/current_cohort_20261030/：完整坐标、方向、覆盖、留鼠、sex对照、检查和置信区间。
- figures/main/Fig4_aging_similarity_draft.pdf：7节点探索性连续热图及层次聚类；不绘制因果网络。
"""
with open("../docs/10_aging_axis_current_20261030.md","w",encoding="utf-8") as handle:handle.write(week10_notes)
notes="""# 第十一周：内部验证与结果稳定性（2026-10-30）

## 完成范围
当前四组织×mct队列，主口径为用户确认的原117个mouse×节点、PC1–7、普通均值。没有重跑或改写08的DE、09的ORA/GSEA及人工清理结果，也不复用旧的无sex DE留鼠结果。
HVG=2000/3000/5000、PC=7/10/20/30、额外最小细胞数=不加门槛/30/50、普通均值/双侧各10%截尾均值，72个全组合全部输出；另有24个精确冻结基设置、2个更严格QC、6个细胞抽样设置、sex调整/仅雄鼠及去复合编号，共107设置、856条节点×设置记录。

HVG使用06权威seurat_v3冻结排序（由counts、未使用年龄标签得到），不是旧注释文件中的过时排序。PCA在与06一致的118105个注释细胞拟合范围重新计算，再提取当前96569个目标细胞；其余组织仅属于冻结表示拟合范围，不是第10/11周目标节点。直接使用安装版Scanpy相同的sklearn PCA/ARPACK后端，random_state=0保持06默认；重抽样seed=2026。不同PCA基不直接比较坐标向量，比较留鼠得分秩相关和各基内部节点余弦相似性。
HVG5000锚点最大坐标误差（允许每PC符号对齐）为0.0005484，基无关余弦最大误差见PCA_basis_invariant_anchor_check.tsv，均在数值容差内。

## 留鼠、bootstrap和QC
全局留一mouse.id时，同一编号从所有组织训练中心同时删除。每节点另输出该mouse的留出投影及删鼠方向与全数据方向余弦。所有年龄缺失/低重复状态保留。
主bootstrap B=500，在全局年龄组内抽mouse.id，并在所有组织共享该mouse的权重；缺失young/old组不补0，输出每个方向/节点对的有效次数、95%分位区间及失败日志。重复抽到同一mouse是合法bootstrap重复，不作为新增动物。
3-M-5/6、3-M-7/8含复合编号，另做剔除两编号及可能7/8与8重叠的共同块bootstrap B=500；不是把不确定池化解释当作已知事实。
QC敏感性在冻结注释singlet及PCA基上施加两组更严格counts/genes阈值（详见QC_threshold_audit.tsv），检验细胞选择影响；没有伪造被上游剔除细胞的mct标签。放宽QC因缺少这些细胞的当前可靠注释，明确记为不可估计。没有声称完整预处理/注释端到端重跑。
随机抽样按每mouse×节点最多100/300细胞、各3个种子，不以细胞为生物学重复。

## 稳定性分级
连续指标优先。以下是事后操作性、非预注册、非显著性检验的分级规则；并保存阈值扫描，不以挑选最有利门槛作为结论：
- 基础未校正方向：端点各≥2个mouse.id，最小LOMO余弦≥0.70，bootstrap余弦95%下界≥0.50，有效bootstrap≥90%，72设置中正式可估计≥80%，且至少80%设置的留鼠得分秩相关≥0.70。
- 总体稳健还要求额外控制（QC/抽样/sex/身份）大多数一致，且sex调整得分秩相关≥0.70。
- 单调年龄趋势另要求原始及sex调整后的留鼠rho≥0.30，并在大多数设置下保持正年龄趋势。
- 节点对的联合稳健性要求两节点总体稳健、配对bootstrap区间不跨0，并在大多数设置中符号一致。区间不跨0不被写成经过FDR校正的多重检验显著。

## 主要结果
基础未校正方向仅Kidney×myeloid通过（bootstrap余弦下界约0.698；LOMO最小余弦约0.976）；其主轴留鼠年龄rho约0.480，但sex加性调整后约0.015，得分秩相关约0.459，故不能作为对sex不敏感的年龄轴。sex调整在每训练折估计性别系数，留出投影先移除female效应，以male为参考；没有用留出样本拟合系数。仅雄鼠对照rho约0.497，两种对照均如实报告，不选择有利一项替代另一项。
因此总体稳健方向为0/7，稳定跨组织节点对为0/21。这里不是证明不存在真实衰老差异，而是当前样本/表示/统计标准下不足以支持稳定共享几何方向。
Kidney×lymphoid年轻细胞总数很少，30/50门槛下没有年轻鼠；Liver×myeloid也因稀疏年轻样本对门槛敏感。Liver×lymphoid继续探索性。spleen/marrow×lymphoid对单鼠高度敏感，留鼠年龄rho为负，不作稳定单调衰老轨迹解释。
Lung两谱系原始cosine约0.444，配对bootstrap区间约[0.046,0.697]，但方向/参数稳健性未通过，不能只据该区间称为稳健共享信号。
宽mct的细胞均值也可能受内部细胞亚型比例影响，不能直接写成细胞内在机制。

## 全部节点指标
""" +table+"""
## 失败与限制
failures.tsv包含172条低重复（exploratory_low_replication）和65条不可估计（包括放宽QC）记录，并非静默跳过或全是软件错误。所有107设置均保存，没有只展示最有利组合。
固定06表示的中心交叉拟合不等于训练折内重训HVG/PCA的完全嵌套验证；未进行外部队列验证。22个mouse.id不是已核实的22只单鼠，复合编号审计必须随结果报告。
09的人工主图排除和功能概念合并属于解释范围整理；本轮PCA敏感性不构成对79个通路逐一重新验证。

## 按任务书处理不稳定结果
跨组织共同方向结论整体降为探索性；不强制构图或建立稳定网络。保留样本级距离作为备用描述，不把鼠对当独立重复。
研究可继续围绕更有证据的节点和独立的sex调整DE/通路结果写作，但不能把本轮相似性热图当成已证实的共同轨迹。是否进一步缩小范围或完整嵌套验证，应由后续任务决定，不在本轮按结果更换主队列。

## 交付
notebooks/11_sensitivity.ipynb内含全部Python逻辑；results/sensitivity/sensitivity_summary.tsv为任务书主入口；current_cohort_20261030子目录保留全部场景、逐鼠、bootstrap、失败、QC、身份、评级与阈值扫描。
figures/supplement/stability_plots.pdf含4页：全部节点区间/年龄趋势、全部72组合、全部21对区间、QC/抽样/sex/身份控制。
历史Notebook、根目录结果及旧Fig4/stability图已备份到.review_backups/weeks10_11_20261030；旧Heart/Harmony审计文档保留为历史，不用于本轮结论。
"""
with open("../docs/result_stability_notes.md","w",encoding="utf-8") as handle:handle.write(notes)
json.dump(dict(primary=params,grid={"HVG":[2000,3000,5000],"PC":[7,10,20,30],"min_cells":[0,30,50],"aggregation":["mean","trimmed_10_percent_each_tail"]},bootstrap=500,seed=2026,PCA_seed=0,QC_scope="stricter annotated-cell selection on frozen PCA, no invented relaxed-QC annotations",criteria_type="operational_not_preregistered",direction_cosine_CI_floor=.5,min_LOMO_cosine_floor=.7,score_rank_correlation_floor=.7,setting_fraction_floor=.8,bootstrap_valid_fraction_floor=.9),
          open("../config/aging_axis_sensitivity_20261030.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
print("Week10/11 reports and complete operational parameter record saved.")


Week10/11 reports and complete operational parameter record saved.


## 交付验收、软件版本与输入输出哈希
逐鼠独立复核投影，核对冻结坐标、完整参数组合和缺失值，记录版本及SHA256。Notebook内容以源代码摘要记录，避免运行输出改变文件字节造成自引用。

In [9]:
# Independent delivery checks and reproducibility record; direct relative I/O only.
import ast
import platform
from datetime import datetime, timezone

checks=[]
def delivery_check(name,condition):
    checks.append(dict(check=name,passed=bool(condition)))
    assert condition,name

audit_scores=pd.read_csv("../results/aging_axis/pca_scores.csv")
audit_means=pd.read_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv",sep="\t")
audit_sim=pd.read_csv("../results/aging_axis/cosine_similarity.csv",index_col=0)
audit_sens=pd.read_csv("../results/sensitivity/sensitivity_summary.tsv",sep="\t")
audit_configs=pd.read_csv("../results/sensitivity/current_cohort_20261030/scenario_summary.tsv",sep="\t")
audit_boot=pd.read_csv("../results/sensitivity/current_cohort_20261030/bootstrap_B500_directions.tsv",sep="\t")
audit_pairs=pd.read_csv("../results/sensitivity/current_cohort_20261030/bootstrap_B500_pairs.tsv",sep="\t")
audit_blocks=pd.read_csv("../results/sensitivity/current_cohort_20261030/composite_block_bootstrap_directions.tsv",sep="\t")
audit_coverage=pd.read_csv("../results/aging_axis/current_cohort_20261030/node_coverage_by_min_cells.tsv",sep="\t")
delivery_check("117_unique_mouse_nodes",len(audit_scores)==117 and not audit_scores.duplicated(["node","mouse_id"]).any())
delivery_check("22_identifier_units_not_asserted_physical_mice",audit_scores.mouse_id.nunique()==22)
delivery_check("primary_7_node_symmetric_cosine",audit_sim.shape==(7,7) and np.allclose(audit_sim,audit_sim.T) and np.allclose(np.diag(audit_sim),1))
delivery_check("exploratory_Liver_lymphoid_excluded","Liver | lymphoid" not in audit_sim.index)
delivery_check("24_month_middle_preserved",audit_scores.loc[audit_scores.age.eq(24),"age_group"].eq("middle").all() and audit_scores.age.eq(24).any())
delivery_check("old_all_male_documented",audit_scores.loc[audit_scores.age_group.eq("old"),"sex"].eq("male").all())
delivery_check("all_107_settings_856_records",len(audit_configs)==107 and len(audit_sens)==856 and audit_sens.setting.nunique()==107)
delivery_check("72_factorial_settings_retained",audit_configs.loc[audit_configs.family.eq("factorial"),"setting"].nunique()==72)
delivery_check("B500_per_primary_direction",len(audit_boot)==3500 and audit_boot.groupby("node").iteration.nunique().eq(500).all())
delivery_check("B500_per_pair",len(audit_pairs)==10500 and audit_pairs.groupby(["node1","node2"]).iteration.nunique().eq(500).all())
delivery_check("B500_overlap_block_control",len(audit_blocks)==3500 and audit_blocks.groupby("node").iteration.nunique().eq(500).all())
delivery_check("missing_bootstrap_not_zero_filled",audit_boot.loc[audit_boot.status.eq("not_estimable"),"cosine_to_baseline"].isna().all())
delivery_check("N30_N50_only_sensitivity",json.load(open("../results/aging_axis/current_cohort_20261030/parameters.json",encoding="utf-8"))["min_cells"]==0 and set(audit_coverage.min_cells)=={0,30,50})
# Independently calculate each held-out centroid and projection from the saved coordinates.
manual_errors=[];unestimable=0
for row in audit_scores.itertuples(index=False):
    train=audit_means.loc[audit_means.node.eq(row.node)&audit_means.mouse_id.ne(row.mouse_id)]
    young=train.loc[train.age_group.eq("young"),[f"PC{i}" for i in range(1,8)]].to_numpy(float)
    old=train.loc[train.age_group.eq("old"),[f"PC{i}" for i in range(1,8)]].to_numpy(float)
    if not len(young) or not len(old):
        unestimable+=1
        delivery_check("heldout_missing_endpoint_"+row.node+"_"+row.mouse_id,pd.isna(row.held_out_score))
        continue
    young_centre=np.mean(young,axis=0);direction=np.mean(old,axis=0)-young_centre
    manual=float(np.dot(np.array([getattr(row,f"PC{i}") for i in range(1,8)])-young_centre,direction)/np.linalg.norm(direction))
    manual_errors.append(abs(manual-row.held_out_score))
delivery_check("independent_all_estimable_heldout_projections",max(manual_errors)<1e-9 and len(manual_errors)+unestimable==117)
# Verify saved mouse means independently against actual frozen cell PC coordinates.
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    audit_obs=read_elem(handle["obs"]);audit_pc=np.asarray(read_elem(handle["obsm/X_pca"]))
for node in NODES:
    row=audit_means.loc[audit_means.node.eq(node)].iloc[0]
    mask=audit_obs["mouse.id"].astype(str).eq(row.mouse_id)&audit_obs.tissue.astype(str).eq(row.tissue)&audit_obs.mct.astype(str).eq(row.mct)
    delivery_check("frozen_cell_mean_"+node,np.allclose(audit_pc[np.asarray(mask),:7].mean(axis=0,dtype=np.float64),row[[f"PC{i}" for i in range(1,8)]].to_numpy(float),atol=5e-5))
delivery_check("canonical_current_score_identity",audit_scores.equals(pd.read_csv("../results/aging_axis/current_cohort_20261030/pca_scores.csv")))
delivery_check("canonical_current_sensitivity_identity",audit_sens.equals(pd.read_csv("../results/sensitivity/current_cohort_20261030/sensitivity_summary.tsv",sep="\t")))
delivery_check("simulation_checks_all_pass",pd.read_csv("../results/aging_axis/current_cohort_20261030/simulation_and_data_checks.tsv",sep="\t").passed.all())
delivery_check("end_to_end_log_complete",open("../results/sensitivity/current_cohort_20261030/full_notebook_execution.log",encoding="utf-8").read().rstrip().endswith("END_TO_END_COMPLETE"))
audit_notebook10=json.load(open("10_aging_axis.ipynb",encoding="utf-8"))
audit_notebook11=json.load(open("11_sensitivity.ipynb",encoding="utf-8"))
for document in [audit_notebook10,audit_notebook11]:
    for cell in document["cells"]:
        if cell["cell_type"]!="code":continue
        source="\n".join(s for s in "".join(cell["source"]).splitlines() if not s.lstrip().startswith("!"))
        tree=ast.parse(source)
        for item in ast.walk(tree):
            if isinstance(item,ast.Import):assert not any(i.name.split(".")[0] in {"os","sys","pathlib"} for i in item.names)
            if isinstance(item,ast.ImportFrom):assert (item.module or "").split(".")[0] not in {"os","sys","pathlib","src"}
            if isinstance(item,ast.Constant) and isinstance(item.value,str):assert not (len(item.value)>2 and item.value[1]==":" and item.value[2] in "/\\")
            if isinstance(item,(ast.Assign,ast.AnnAssign)) and isinstance(item.value,ast.Constant) and isinstance(item.value.value,str):assert not item.value.value.startswith("../")
delivery_check("notebooks_inline_relative_IO_no_os_sys_path",True)
delivery_check("source_module_matches_notebook_functions","".join(audit_notebook10["cells"][1]["source"]) in open("../src/aging_axis.py",encoding="utf-8").read())
pd.DataFrame(checks).to_csv("../results/sensitivity/current_cohort_20261030/delivery_validation.tsv",sep="\t",index=False)

completion_notes="""# 第10/11周交付验收（2026-10-30）

| 任务书事项 | 本轮结果 | 边界 |
|---|---|---|
| 鼠级PCA、中心差方向、长度、投影及连续年龄趋势 | 已输出117个鼠×节点、8节点方向/长度；正式相似性7节点 | 22个mouse.id含复合编号；不是已核实22只物理单鼠 |
| 模拟及真实数据检查、留出投影 | 解析模拟通过；逐行独立复核117行中的可估计投影 | HVG/PCA表示固定于06；不是全嵌套表示学习验证 |
| 连续相似性热图和层次聚类 | Fig4完成；21对原始余弦及配对区间保存 | Liver淋巴系继续探索性；高余弦不能替代稳健性 |
| LOMO、HVG、PC、细胞门槛、均值方法 | 全72组合及额外控制，共107设置、856节点记录 | 用户明确117为主，30/50仅敏感性；所有失败均保留 |
| 鼠级bootstrap B=500 | 共享跨组织鼠权重；另做可能重叠编号的共同块控制 | 年龄分层；缺失端点不补0；区间不是FDR检验 |
| QC与细胞抽样敏感性 | 两组更严格QC，100/300细胞×3种子 | 放宽QC因缺少可靠现行mct标签不可估计，已记录 |
| 不稳定结果降级 | 总体稳健方向0/7、联合稳健节点对0/21 | 仅肾髓系未校正几何方向较稳定；sex调整趋势敏感 |
| 备用结果 | 样本级距离及年龄组摘要 | 不把鼠对或细胞当独立生物学重复 |
| 可复现交付 | 两本Notebook内嵌Python、相对地址、模块同步、软件版本/输入输出SHA256 | 本轮不重跑DE/GSEA，不代表逐通路验证 |

主分析和所有敏感性分析均已执行，不以任务书的期望阳性结论作为验收要求。当前数据不足以把原始相似性表述为稳健共同衰老轨迹。

图件检查：Fig4及4页稳定性PDF已渲染并逐页检查；文字、色标和图注没有相互遮挡。完整运行日志见results/sensitivity/current_cohort_20261030/full_notebook_execution.log，机器验收见同目录delivery_validation.tsv。
"""
with open("../docs/weeks10_11_delivery_20261030.md","w",encoding="utf-8") as handle:handle.write(completion_notes)
existing_log=open("../analysis_log.md",encoding="utf-8").read()
if "## 2026-10-30：当前队列第10/11周重做" not in existing_log:
    with open("../analysis_log.md","a",encoding="utf-8") as handle:
        handle.write("\n\n## 2026-10-30：当前队列第10/11周重做\n\n用户确认08原117鼠×节点为主分析，30/50细胞仅敏感性。使用当前四组织、两mct、24月middle、30月old及06冻结PCA；不复用历史Heart/Harmony队列和旧第11周稳健结论。完整执行两本Notebook，72全组合+额外控制共107设置/856记录，LOMO、B500配对鼠bootstrap及复合编号控制全部输出。7正式节点、21对；仅Kidney髓系未校正方向通过基础几何规则，sex标准化留出趋势弱，总体稳健方向0/7、联合稳健节点对0/21，均按探索性解释。当前主入口results/weeks10_11_latest.json；报告docs/result_stability_notes.md，图件Fig4_aging_similarity_draft.pdf和supplement/stability_plots.pdf。旧Notebook/根目录结果/图件保留在.review_backups/weeks10_11_20261030。此前日志中的旧队列第10/11周数字为历史记录，不代表本轮结果。\n")

# Stream file contents rather than storing huge H5AD payloads in memory.
def sha256_stream(handle):
    digest=hashlib.sha256()
    with handle:
        while True:
            block=handle.read(8*1024*1024)
            if not block:break
            digest.update(block)
    return digest.hexdigest()

def notebook_source_digest(document):
    payload=[dict(cell_type=c["cell_type"],source="".join(c["source"])) for c in document["cells"]]
    return hashlib.sha256(json.dumps(payload,ensure_ascii=False,sort_keys=True).encode("utf-8")).hexdigest()

versions={package:importlib.metadata.version(package) for package in ["numpy","pandas","scipy","anndata","h5py","scikit-learn","scanpy","threadpoolctl","matplotlib"]}
versions["Python"]=platform.python_version()
input_sha256={
    "data_processed/final_data.h5ad":sha256_stream(open("../data_processed/final_data.h5ad","rb")),
    "data_processed/pseudobulk.h5ad":sha256_stream(open("../data_processed/pseudobulk.h5ad","rb")),
    "data_interim/cell_type_annotated.h5ad":sha256_stream(open("../data_interim/cell_type_annotated.h5ad","rb")),
    "本科毕业论文_多组织衰老共同轨迹_18周执行计划_暴璐玮.docx":sha256_stream(open("../本科毕业论文_多组织衰老共同轨迹_18周执行计划_暴璐玮.docx","rb")),
}
output_sha256={
    "analysis_log.md":sha256_stream(open("../analysis_log.md","rb")),
    "config/aging_axis_sensitivity_20261030.json":sha256_stream(open("../config/aging_axis_sensitivity_20261030.json","rb")),
    "docs/10_aging_axis_current_20261030.md":sha256_stream(open("../docs/10_aging_axis_current_20261030.md","rb")),
    "docs/result_stability_notes.md":sha256_stream(open("../docs/result_stability_notes.md","rb")),
    "docs/weeks10_11_delivery_20261030.md":sha256_stream(open("../docs/weeks10_11_delivery_20261030.md","rb")),
    "figures/main/Fig4_aging_similarity_draft.pdf":sha256_stream(open("../figures/main/Fig4_aging_similarity_draft.pdf","rb")),
    "figures/main/Fig4_aging_similarity_draft.png":sha256_stream(open("../figures/main/Fig4_aging_similarity_draft.png","rb")),
    "figures/supplement/stability_plots.pdf":sha256_stream(open("../figures/supplement/stability_plots.pdf","rb")),
    "figures/supplement/stability_plots.png":sha256_stream(open("../figures/supplement/stability_plots.png","rb")),
    "results/aging_axis/cosine_similarity.csv":sha256_stream(open("../results/aging_axis/cosine_similarity.csv","rb")),
    "results/aging_axis/current_cohort_20261030/aging_directions.csv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/aging_directions.csv","rb")),
    "results/aging_axis/current_cohort_20261030/aging_similarity.pdf":sha256_stream(open("../results/aging_axis/current_cohort_20261030/aging_similarity.pdf","rb")),
    "results/aging_axis/current_cohort_20261030/all_mouse_node_PCA.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/all_mouse_node_PCA.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/cosine_similarity.csv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/cosine_similarity.csv","rb")),
    "results/aging_axis/current_cohort_20261030/held_out_age_scores.pdf":sha256_stream(open("../results/aging_axis/current_cohort_20261030/held_out_age_scores.pdf","rb")),
    "results/aging_axis/current_cohort_20261030/held_out_age_scores.png":sha256_stream(open("../results/aging_axis/current_cohort_20261030/held_out_age_scores.png","rb")),
    "results/aging_axis/current_cohort_20261030/leave_one_mouse_out.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/leave_one_mouse_out.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/mouse_PCA_trimmed.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/mouse_PCA_trimmed.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/node_coverage_by_min_cells.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/node_coverage_by_min_cells.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/node_summary.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/node_summary.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/outside_current_DE_cohort.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/outside_current_DE_cohort.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/pairwise_cosine_CI.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/pairwise_cosine_CI.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/parameters.json":sha256_stream(open("../results/aging_axis/current_cohort_20261030/parameters.json","rb")),
    "results/aging_axis/current_cohort_20261030/pca_scores.csv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/pca_scores.csv","rb")),
    "results/aging_axis/current_cohort_20261030/sample_distance_summary.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/sample_distance_summary.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/sex_sensitivity.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/sex_sensitivity.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/simulation_and_data_checks.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/simulation_and_data_checks.tsv","rb")),
    "results/aging_axis/current_cohort_20261030/within_node_sample_distances.tsv":sha256_stream(open("../results/aging_axis/current_cohort_20261030/within_node_sample_distances.tsv","rb")),
    "results/aging_axis/node_summary.tsv":sha256_stream(open("../results/aging_axis/node_summary.tsv","rb")),
    "results/aging_axis/pca_scores.csv":sha256_stream(open("../results/aging_axis/pca_scores.csv","rb")),
    "results/pca/mouse_level_pca.csv":sha256_stream(open("../results/pca/mouse_level_pca.csv","rb")),
    "results/sensitivity/current_cohort_20261030/delivery_validation.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/delivery_validation.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/HVG5000_anchor_check.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/HVG5000_anchor_check.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/HVG_PCA_run_log.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/HVG_PCA_run_log.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/PCA_HVG2000.npy":sha256_stream(open("../results/sensitivity/current_cohort_20261030/PCA_HVG2000.npy","rb")),
    "results/sensitivity/current_cohort_20261030/PCA_HVG3000.npy":sha256_stream(open("../results/sensitivity/current_cohort_20261030/PCA_HVG3000.npy","rb")),
    "results/sensitivity/current_cohort_20261030/PCA_HVG5000.npy":sha256_stream(open("../results/sensitivity/current_cohort_20261030/PCA_HVG5000.npy","rb")),
    "results/sensitivity/current_cohort_20261030/PCA_basis_invariant_anchor_check.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/PCA_basis_invariant_anchor_check.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/PCA_source_cell_metadata.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/PCA_source_cell_metadata.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/QC_threshold_audit.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/QC_threshold_audit.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/all_scenario_held_out_scores.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/all_scenario_held_out_scores.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/bootstrap_B500_directions.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/bootstrap_B500_directions.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/bootstrap_B500_pairs.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/bootstrap_B500_pairs.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/bootstrap_direction_CI.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/bootstrap_direction_CI.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/bootstrap_failures.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/bootstrap_failures.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/composite_block_bootstrap_directions.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/composite_block_bootstrap_directions.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/composite_block_bootstrap_pairs.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/composite_block_bootstrap_pairs.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/failures.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/failures.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/full_notebook_execution.log":sha256_stream(open("../results/sensitivity/current_cohort_20261030/full_notebook_execution.log","rb")),
    "results/sensitivity/current_cohort_20261030/genes_HVG2000.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/genes_HVG2000.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/genes_HVG3000.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/genes_HVG3000.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/genes_HVG5000.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/genes_HVG5000.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/global_lomo_pairwise.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/global_lomo_pairwise.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/mouse_identity_audit.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/mouse_identity_audit.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/node_stability_verdict.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/node_stability_verdict.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/pair_similarity_stability.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/pair_similarity_stability.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/pairwise_cosine_CI.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/pairwise_cosine_CI.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/preflight_resolution_log.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/preflight_resolution_log.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/representation_execution.log":sha256_stream(open("../results/sensitivity/current_cohort_20261030/representation_execution.log","rb")),
    "results/sensitivity/current_cohort_20261030/scenario_pairwise_cosines.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/scenario_pairwise_cosines.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/scenario_summary.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/scenario_summary.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/scenarios_execution.log":sha256_stream(open("../results/sensitivity/current_cohort_20261030/scenarios_execution.log","rb")),
    "results/sensitivity/current_cohort_20261030/sensitivity_summary.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/sensitivity_summary.tsv","rb")),
    "results/sensitivity/current_cohort_20261030/stability_threshold_scan.tsv":sha256_stream(open("../results/sensitivity/current_cohort_20261030/stability_threshold_scan.tsv","rb")),
    "results/sensitivity/sensitivity_summary.tsv":sha256_stream(open("../results/sensitivity/sensitivity_summary.tsv","rb")),
    "src/aging_axis.py":sha256_stream(open("../src/aging_axis.py","rb")),
}

manifest=dict(created_utc=datetime.now(timezone.utc).isoformat(),analysis_date_local="2026-10-30",analysis_status="complete_with_explicit_exploratory_downgrades",primary_mouse_nodes=117,mouse_identifier_units=22,primary_nodes=7,pairs=21,settings=107,factorial_settings=72,node_setting_records=856,bootstrap_B=500,overall_robust_nodes=0,consistent_node_pairs=0,criteria="operational_posthoc_not_significance_testing",python_logic="inline_notebooks_no_external_analysis_imports",versions=versions,input_sha256=input_sha256,output_sha256=output_sha256,notebook_analysis_source_sha256={"10_aging_axis.ipynb":notebook_source_digest(audit_notebook10),"11_sensitivity.ipynb":notebook_source_digest(audit_notebook11)},validation_checks=len(checks),all_checks_passed=True,limitations=["06 frozen PCA/HVG; cross-fitting of centres/directions is conditional on that representation, not fully nested","old mice all male; female aging/age-sex interaction not identified","composite mouse IDs not asserted individual physical animals","stricter QC only; relaxed-QC current labels unavailable","no independent external cohort or per-pathway revalidation"])
json.dump(manifest,open("../results/sensitivity/current_cohort_20261030/manifest.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
json.dump(manifest,open("../results/aging_axis/current_cohort_20261030/manifest.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
json.dump(dict(date="2026-10-30",week10="results/aging_axis/current_cohort_20261030",week11="results/sensitivity/current_cohort_20261030",week10_notebook="notebooks/10_aging_axis.ipynb",week11_notebook="notebooks/11_sensitivity.ipynb",main_figure="figures/main/Fig4_aging_similarity_draft.pdf",sensitivity_figure="figures/supplement/stability_plots.pdf",notes="docs/result_stability_notes.md",manifest="results/sensitivity/current_cohort_20261030/manifest.json",interpretation="exploratory_shared_geometry; no pairs passed joint operational robustness criteria"),open("../results/weeks10_11_latest.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
print("All",len(checks),"delivery checks passed; versions and",len(input_sha256),"input /",len(output_sha256),"output SHA256 hashes saved.")


All 29 delivery checks passed; versions and 4 input / 67 output SHA256 hashes saved.


## 2026-10-30交付补齐 · 当前统一参数、元数据说明与项目入口
独立读取已有对象导出，不重跑上游拟合；运行目录为notebooks。历史版本保留在整理备份。


In [14]:
import json
import hashlib
import h5py
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy import sparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42

import yaml
import importlib.metadata
! powershell.exe -NoProfile -Command "New-Item -ItemType Directory -Force '../environment','../results/workspace_audit_20261030' | Out-Null"
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    frozen_obs=read_elem(handle["obs"]);frozen_uns=read_elem(handle["uns"])
de_record=json.load(open("../all_results.manifest.json",encoding="utf-8"))
gsea_record=json.load(open("../results/enrichment/gseapy_gobp_gsea_20261029/manifest.json",encoding="utf-8"))
axis_record=json.load(open("../config/aging_axis_sensitivity_20261030.json",encoding="utf-8"))
qc_record=pd.read_csv("../results/qc/qc_parameters_draft.tsv",sep="\t")
configuration={
"project":{"name":"多组织衰老共同轨迹（Tabula Muris Senis）","current_export_date":"2026-10-30","directory_cleanup_completed":"2026-10-31","status":"current frozen analysis; no refit during directory cleanup"},
"cohort":{"tissues":["Kidney","Liver","Lung","spleen/marrow"],"mct":["lymphoid","myeloid"],"frozen_cells":96569,"observed_mouse_nodes":120,"primary_mouse_nodes":117,"mouse_identifier_units":22,"unit_note":"mouse.id may be composite; not22 verified physical animals","exploratory_node":"Liver | lymphoid"},
"age_groups":{"young":[1,3],"middle":[18,21,24],"old":[30],"authority":"06 current frozen labels"},
"qc":{"per_tissue":qc_record.set_index("tissue")[["min_counts","max_counts","min_genes","max_genes"]].astype(int).to_dict("index"),"doublet_method":"scDblFinder","doublet_seed":123,"mt_filter_applied":False,"matched_mt_prefix_genes":int(qc_record.mt_prefix_genes.iloc[0])},
"representation":{"normalization_target_sum":10000,"log1p":True,"scaled":False,"HVG":{"flavor":"seurat_v3","layer":"counts","n_top_genes":5000},"pca":{"n_comps":50,"solver":"arpack","random_state":0,"fit_scope":"118105 annotated cells before target selection"},"neighbors":{"n_neighbors":15,"n_pcs":6,"random_state":2026},"leiden":{"resolution":0.5,"random_state":2026,"flavor":"igraph","n_iterations":2},"umap":{"min_dist":0.3,"init_pos":"paga","random_state":0},"Harmony":False},
"pseudobulk":{"h5ad_rows":375,"groups":["mouse.id","tissue","cell_type"],"layer":"counts","aggregation":"sum","min_cells":3,"min_counts":200,"DE_aggregation":["mouse.id","tissue","mct"],"DE_mouse_nodes":117,"counts_tsv_orientation":"genes x375 unique sample_id columns"},
"de":{"tool":"PyDESeq2","version":"0.5.4","actual_node_design":"~sex+age_group","contrast":"old vs young","middle_in_model":True,"alpha":.05,"confidence_level":.95,"n_cpus":1,"Cook_handling":"unchanged from08","independent_filtering":True,"expression_filtered_genes":9165,"filter_parameters":de_record["parameters"],"exploratory_policy":"effect and CI; no P/FDR or enrichment"},
"ORA":{"tool":"GSEApy","foreground":"FDR<0.05 split by log2FC sign; no effect-size threshold","background":"all9165 expression-filtered genes","min_size":10,"max_size":500,"eps":None,"correction":"BH over all eligible terms including zero hits within node x direction"},
"GSEA":gsea_record["parameters"],
"final_GSEA_display":{"cutoff":.7,"semantic_measure":"Wang","robustness":"main and strict-rank BH-FDR<0.05; concordant NES signs","manual_context_review":True,"GO_terms":79,"significant_records":150,"up":101,"down":49,"classification":"existing reviewed classes, unchanged","selection_is_not_proof_of_false_positive_elimination":True},
"aging_axis_and_sensitivity":axis_record,
"optional":{"scVI":"not performed for current cohort","UCE":"not performed for current cohort","weeks12_18":"future work; legacy outputs are not current completion"}
}
yaml.safe_dump(configuration,open("../config/analysis_parameters_v1.yaml","w",encoding="utf-8"),allow_unicode=True,sort_keys=False)
# Preserve old provenance while registering the final current display explicitly.
state=json.load(open("../results/sex_ora_latest.json",encoding="utf-8"))
state["active_gsea_context_curated"]="results/enrichment/gsea_context_curated_20261029"
state["active_gsea_figure"]="results/enrichment/gsea_context_curated_20261029/figures/GSEA_context_curated_0.7_vertical_clusters.pdf"
state["canonical_current_GSEA"]="results/enrichment/gsea_all_nodes.tsv"
state["canonical_current_pathway_matrix"]="results/enrichment/pathway_matrix.csv"
state["current_state_updated"]="2026-10-30; directory delivery update only; no enrichment rerun"
json.dump(state,open("../results/sex_ora_latest.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
with open("../docs/analysis_plan_v1.md","w",encoding="utf-8") as handle:
    handle.write("""# 当前分析范围及已完成方案（2026-10-30）
本文件替代旧Heart/内皮/巨噬细胞计划，记录已执行分析，不是新的预注册。
四组织Kidney、Liver、Lung、spleen/marrow，两mct为lymphoid/myeloid。年龄1/3月young，18/21/24月middle，30月old。
冻结96569细胞、120个有细胞鼠×节点；沿用07样本规则进入08的主队列为117，含22个mouse.id编号。复合编号及潜在重叠已在11审计，不能声称22只已核实独立单鼠。
原始counts按鼠×组织×cell_type求和得到375条07记录，08再按mct合并；过滤后9165基因，节点内PyDESeq2~sex+age_group，保留middle，old vs young。Liver淋巴系探索性，只留效应与区间。
ORA背景为完整过滤基因，前景FDR<0.05按效应方向拆分；本地GOBP与背景交集后10–500，所有合格集合BH，eps不适用。
GSEA为Wald排序及已完成的严格排序敏感性，eps=0；最终0.7显示为79GO/150条稳健记录，含人工背景审查与概念整理，不表示已证明删除全部假阳性。
PCA为冻结log1p表达、5000HVG、未scale、无Harmony；118105注释细胞拟合后选择目标细胞。第10周主轴PC1–7；117队列为主，30/50仅敏感性；第11周107设置和B500均保留。
当前总体稳健方向0/7、联合稳健节点对0/21，按探索性解释，不为获得共同轨迹而更换主队列。
所有old为雄性，不推断雌鼠衰老或年龄×性别交互；mct均值可受亚型比例影响。QC没有执行线粒体阈值筛选，当前符号前缀未匹配到线粒体基因，必须写入限制。
清理只新增交付、归档历史并删除可重建过程文件，不改冻结对象、DE或富集原始统计。第12–18周尚未作为当前队列任务执行。
""")
with open("../docs/go_no_go_week2.md","w",encoding="utf-8") as handle:
    handle.write("""# 第2周可行性与最终范围（2026-10-30交付更新）
候选checked数据144809细胞、6组织标签；Marrow与Spleen合并。results/qc中的覆盖表使用实际观测组合，不包含展开产生的零细胞行。
当前最终选择Kidney、Liver、Lung、spleen/marrow及lymphoid/myeloid，96569细胞、120个观测鼠×节点，117进入主分析。早期候选阶段与最终队列不能混用。
有可追踪mouse.id和原始整数counts，因此可开展鼠级描述及节点DE；但Liver淋巴系仅1个young，降为探索性。其余节点也有稀疏样本，不声称全部达到30细胞：117队列为用户确认主分析，30/50仅敏感性。
22个mouse.id含复合编号；old全部雄性。当前证据不足以确认稳健跨组织共同PCA衰老方向，仍允许如实报告组织/谱系特异差异与限制。
""")
with open("../docs/metadata_dictionary.md","w",encoding="utf-8") as handle:
    handle.write("""# 当前数据与字段字典（2026-10-30）
| 对象 | 口径 | 表达矩阵 |
|---|---|---|
| data_raw官方7文件 | 原始只读；先前下载版本保留 | 官方.X及.raw.X的含义分别核查，不以文件扩展名推断counts |
| data_interim/TMS_checked.h5ad | 候选144809细胞、6组织标签 | 由raw counts导出，本轮分块核查非负整数 |
| data_interim/cell_type_annotated.h5ad | 118105个注释细胞；06表示拟合范围 | X为log1p，layers/counts为整数counts |
| data_processed/final_data.h5ad | 当前四组织两mct，96569细胞 | X为log1p，layers/counts为整数counts；冻结PCA/UMAP |
| data_processed/pseudobulk.h5ad | 375条鼠×组织×cell_type记录 | X为原始整数求和counts |
mouse.id是原始编号，可能含斜杠复合编号；导出表统一为mouse_id，保留原值，不清洗合并身份。
age是月龄，06冻结young=1/3、middle=18/21/24、old=30；sex为male/female。旧参数中的24月old已不适用。
tissue保留Kidney/Liver/Lung/spleen/marrow；cell_ontology_class为原始标签，cell_type为05注释细分类，mct为统一谱系。
07 metadata的sample_id沿用H5AD obs_names，唯一且与counts列顺序一致；source_sample为原mouse×组织标识，不具有375行唯一性。
psbulk_cells与psbulk_counts分别为该cell_type聚合的细胞数和文库counts；08按mouse_id×tissue×mct聚合成117行。FDR缺失保留缺失，不转换为非显著或零。
final_cohort_matrix覆盖120个有细胞分组，in_primary_117指示117个主分析分组，exploratory_node指示Liver淋巴系。
""")
with open("../README.md","w",encoding="utf-8") as handle:
    handle.write("""# 多组织衰老共同轨迹：当前工作入口
分析版本：2026-10-30；目录整理完成：2026-10-31。当前队列为Kidney、Liver、Lung、spleen/marrow的lymphoid/myeloid；96569细胞、120个观测鼠×节点、117个主分析鼠×节点。22个mouse.id含复合编号，不等于已核实22只单鼠。24月middle，30月old。

## 当前入口
- 参数：config/analysis_parameters_v1.yaml。
- DE来源：all_results.manifest.json、results/de、results/de_runs/20261028T080159Z_sex_adjusted。
- ORA/GSEA来源和最终显示：results/sex_ora_latest.json；最终0.7为results/enrichment/gsea_context_curated_20261029，79GO、150条显著记录。
- 第10/11周：results/weeks10_11_latest.json；主队列117，30/50仅敏感性。
- 主图Fig2/Fig3已按当前对象更新；Fig4和supplement/stability_plots为2026-10-30分析。
- 任务书对照及整理报告：docs/taskbook_compliance_matrix.tsv、docs/workspace_cleanup_20261030.md。
- 机器审计：results/workspace_audit_20261030。历史文件位于archive/legacy_20261030；已有.review_backups保留原位。

## 运行方式与数据契约
Notebook工作目录为notebooks。01–11为现行周次入口，新增“2026-10-30交付补齐”章节可独立读取现存对象，仅重导出图表，不重新拟合DE/GSEA；不要为了查看结果从头执行全部分析。
任务书00 smoke test对应现有01；edgeR实现由已批准的PyDESeq2替代，不补造R脚本。当前冻结交付采用final_data.h5ad，旧main_qc已归档，不再存大型重复对象。
pseudobulk_counts.tsv行为20138基因、列为375个sample_id，metadata同序；08按mct汇总得到117行。pathway_matrix.csv为79×7原始NES，pathway_display_mask.csv决定最终显示，不能把未显示补成0。
第2周coverage表属于候选144809细胞；最终范围见results/qc/final_cohort_matrix.csv，不能跨阶段混用。

## 删除文件的恢复
原始数据、TMS_checked汇总、7个standardized交付、filtered_doublet、normalized、neighbored、dimensionality_reduced、cell_type_annotated、final_data及pseudobulk保留。
需要恢复七个TMS组织提取文件：在02执行原始数据读取及raw提取保存单元。
需要恢复六个组织checked文件：03从保留的TMS_checked执行分组织写出单元。
需要恢复filtered：先恢复六个checked输入，再执行04的读取、QC指标、组织阈值和filtered写出单元；到此停止，不覆盖已冻结的后续结果。详见删除记录中的producer/recovery。
旧04仍有工作区绝对输入地址，03仍含Path写法，均在整理报告中登记；本轮新增交付代码均使用直接相对I/O，不以目录整理为由改写旧预处理算法。

## 解释与后续
当前DE含sex加性项，old全部雄性，不推断雌鼠衰老或交互；Liver淋巴系探索性。总体稳健PCA方向0/7、节点对0/21，共同轨迹仍为探索性。
线粒体前缀未匹配基因，且04没有实际执行mt阈值，不能声称完成有效线粒体比例筛选。
第12–18周属于后续任务；旧稿、旧Fig5及旧总结不代表当前队列已完成后续周次。软件环境清单位于environment；本轮不安装或升级依赖。
""")
versions={package:importlib.metadata.version(package) for package in ["numpy","pandas","scipy","anndata","h5py","scikit-learn","scanpy","gseapy","pydeseq2","decoupler","matplotlib","PyYAML"]}
json.dump(versions,open("../environment/python_versions_20261030.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
print("Current parameters, cohort/metadata notes, README and Python version record updated; no analysis refit.")


Current parameters, cohort/metadata notes, README and Python version record updated; no analysis refit.


## 目录整理交付验收
独立核对导出与冻结对象，作为归档和删除前的验收门槛。


In [15]:
import json
import hashlib
import h5py
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy import sparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42

import ast
import yaml
checks=[]
def check_delivery(name,passed):
    checks.append(dict(check=name,passed=bool(passed)))
    assert passed,name
meta=pd.read_csv("../data_processed/pseudobulk_metadata.tsv",sep="\t")
counts=pd.read_csv("../data_processed/pseudobulk_counts.tsv",sep="\t",index_col=0)
with h5py.File("../data_processed/pseudobulk.h5ad","r") as handle:
    pb_obs=read_elem(handle["obs"]);pb_var=read_elem(handle["var"]);pb_counts=np.asarray(read_elem(handle["X"]))
check_delivery("pseudobulk375_samples_20138_genes",counts.shape==(20138,375) and len(meta)==375)
check_delivery("pseudobulk_ID_order_exact",counts.columns.tolist()==meta.sample_id.tolist()==pb_obs.index.astype(str).tolist() and counts.index.tolist()==pb_var.index.astype(str).tolist())
check_delivery("pseudobulk_counts_exact_integer_H5AD",np.array_equal(counts.to_numpy().T,pb_counts) and (counts.to_numpy()>=0).all() and np.equal(counts.to_numpy(),np.rint(counts.to_numpy())).all())
check_delivery("pseudobulk117_mouse_nodes",meta[["mouse_id","tissue","mct"]].drop_duplicates().shape[0]==117)
coverage=pd.read_csv("../results/qc/sample_coverage.csv")
types=pd.read_csv("../results/qc/celltype_coverage.csv")
check_delivery("coverage_actual_observed_groups",len(coverage)==80 and coverage.n_cells.gt(0).all() and coverage.n_cells.sum()==144809)
check_delivery("celltype_candidate_coverage",types.n_cells.gt(0).all() and types.n_cells.sum()==144809)
check_delivery("mouse_metadata_consistent",coverage.groupby("mouse_id")[["age_months","sex"]].nunique().le(1).all().all())
audit=pd.read_csv("../results/qc/filtering_audit.csv")
stage_columns=["candidate","counts_genes_filtered","singlet","annotated","frozen_target"]
check_delivery("filtering_audit_totals",audit[stage_columns].sum().tolist()==[144809,141471,133745,118105,96569])
check_delivery("filtering_audit_monotone_membership_counts",np.diff(audit[stage_columns].to_numpy(),axis=1).max()<=0)
cohort=pd.read_csv("../results/qc/final_cohort_matrix.csv")
check_delivery("120_observed117_primary",len(cohort)==120 and cohort.in_primary_117.sum()==117 and cohort.n_cells.sum()==96569)
check_delivery("primary_unique",not cohort.duplicated(["mouse_id","tissue","mct"]).any())
check_delivery("exploratory_correct",cohort.exploratory_node.equals(cohort.tissue.eq("Liver")&cohort.mct.eq("lymphoid")))
qc=pd.read_csv("../results/qc/qc_parameters_draft.tsv",sep="\t")
check_delivery("no_fictitious_mitochondrial_filter",qc.mt_prefix_genes.eq(0).all() and qc.mt_filter_applied.eq(False).all())
lib=pd.read_csv("../results/pseudobulk/library_qc.csv")
check_delivery("library_QC_exact",len(lib)==375 and np.array_equal(lib.library_size_from_integer_counts.to_numpy(),pb_counts.sum(axis=1)) and np.array_equal(lib.n_detected_genes.to_numpy(),(pb_counts>0).sum(axis=1)))
original=pd.read_csv("../results/enrichment/gseapy_gobp_gsea_20261029/gsea_all_nodes.tsv",sep="\t",float_precision="round_trip")
canonical=pd.read_csv("../results/enrichment/gsea_all_nodes.tsv",sep="\t",float_precision="round_trip")
check_delivery("canonical_full_GSEA_exact_original",original.equals(canonical) and len(canonical)==26978)
retained=pd.read_csv("../results/enrichment/gsea_context_curated_20261029/curated_significant_0.7.tsv",sep="\t",float_precision="round_trip")
check_delivery("final79_GO150_records",len(retained)==150 and retained.GOID.nunique()==79)
check_delivery("retained_joint_significance",retained.BH_FDR.lt(.05).all() and retained.strict_BH_FDR.lt(.05).all() and (np.sign(retained.NES)==np.sign(retained.strict_NES)).all())
check_delivery("retained_direction_counts",(retained.NES>0).sum()==101 and (retained.NES<0).sum()==49)
canonical_matrix=pd.read_csv("../results/enrichment/pathway_matrix.csv",index_col=0,float_precision="round_trip")
mask=pd.read_csv("../results/enrichment/pathway_display_mask.csv",index_col=0)
source_matrix=pd.read_csv("../results/enrichment/gsea_context_curated_20261029/NES_matrix_0.7.tsv",sep="\t",header=[0,1],index_col=0,float_precision="round_trip").loc[canonical_matrix.index]
check_delivery("canonical_NES79x7_matches_legacy_display_within_serialization_tolerance",canonical_matrix.shape==(79,7) and np.allclose(canonical_matrix.to_numpy(),source_matrix.to_numpy(),rtol=0,atol=1e-14,equal_nan=True))
check_delivery("mask150_no_exploratory",mask.to_numpy().sum()==150 and not any("Liver | lymphoid"==c for c in mask.columns))
mapping=retained[["GOID","pathway"]].drop_duplicates().set_index("GOID").pathway
raw_lookup=original.set_index(["tissue","mct","pathway"])
nes_errors=[]
for go_id in canonical_matrix.index:
    for column in canonical_matrix.columns:
        tissue,mct=column.split(" | ")
        expected=raw_lookup.loc[(tissue,mct,mapping[go_id]),"NES"]
        actual=canonical_matrix.loc[go_id,column]
        nes_errors.append((pd.isna(expected) and pd.isna(actual)) or expected==actual)
check_delivery("NES_restored_from_each_node_original",all(nes_errors))
merged=retained.merge(original,on=["tissue","mct","pathway"],suffixes=("_display","_raw"),validate="many_to_one")
serialization_audit=merged[["GOID","tissue","mct","pathway","NES_display","NES_raw","BH_FDR_display","BH_FDR_raw"]].copy()
serialization_audit["NES_absolute_text_roundoff"]=abs(serialization_audit.NES_display-serialization_audit.NES_raw)
serialization_audit["BH_FDR_absolute_text_roundoff"]=abs(serialization_audit.BH_FDR_display-serialization_audit.BH_FDR_raw)
serialization_audit.to_csv("../results/workspace_audit_20261030/legacy_display_float_serialization_audit.tsv",sep="\t",index=False)
check_delivery("retained_matches_original_with_recorded_legacy_text_roundoff",len(merged)==150 and serialization_audit.NES_absolute_text_roundoff.le(1e-14).all() and serialization_audit.BH_FDR_absolute_text_roundoff.le(1e-15).all() and merged.leading_edge_genes_display.equals(merged.leading_edge_genes_raw))
explore=pd.read_csv("../results/de/Liver_lymphoid_age_DE.tsv",sep="\t")
check_delivery("exploratory_P_FDR_remain_missing",explore.PValue.isna().all() and explore.FDR.isna().all())
params=yaml.safe_load(open("../config/analysis_parameters_v1.yaml",encoding="utf-8"))
check_delivery("current_parameters_cohort_and_age",params["cohort"]["primary_mouse_nodes"]==117 and {g:params["age_groups"][g] for g in ["young","middle","old"]}=={"young":[1,3],"middle":[18,21,24],"old":[30]} and params["representation"]["Harmony"] is False)
check_delivery("current_DE_parameters",params["de"]["actual_node_design"]=="~sex+age_group" and params["de"]["expression_filtered_genes"]==9165)
check_delivery("primary_no_extra_cell_threshold",params["aging_axis_and_sensitivity"]["primary"]["min_cells"]==0)
qa=json.load(open("../results/workspace_audit_20261030/pdf_qa.json",encoding="utf-8"))
check_delivery("all5_PDFs10_pages",sum(v["pages"] for v in qa.values())==10 and len(qa)==5)
check_delivery("Fig3_PDF_actual150_stars",qa["Fig3"]["stars_excluding_one_footnote_per_page"]==150)
r_record=open("../environment/R_sessionInfo_20261030.txt",encoding="utf-8-sig").read()
check_delivery("R_semantic_and_doublet_packages_load",r_record.count("namespace load OK")==5 and "LOAD FAILED" not in r_record)
# Inspect only newly appended export sections; existing upstream style is retained and documented.
documents=[json.load(open("02_TMS_data_check_and_extract.ipynb",encoding="utf-8")),json.load(open("04_QC.ipynb",encoding="utf-8")),json.load(open("05_cell_type_annotation_and_sample_selection.ipynb",encoding="utf-8")),json.load(open("07_pseudobulk_build.ipynb",encoding="utf-8")),json.load(open("08_DE.ipynb",encoding="utf-8")),json.load(open("09_GOBP_ORA.ipynb",encoding="utf-8")),json.load(open("10_aging_axis.ipynb",encoding="utf-8")),json.load(open("11_sensitivity.ipynb",encoding="utf-8"))]
for book in documents:
    reviewing=False
    for cell in book["cells"]:
        if cell["cell_type"]=="markdown" and "2026-10-30交付补齐" in "".join(cell["source"]):reviewing=True
        if not reviewing or cell["cell_type"]!="code":continue
        source="\n".join(line for line in "".join(cell["source"]).splitlines() if not line.lstrip().startswith("!"))
        tree=ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node,ast.Import):assert not any(x.name.split(".")[0] in {"os","sys","pathlib"} for x in node.names)
            if isinstance(node,ast.ImportFrom):assert (node.module or "").split(".")[0] not in {"os","sys","pathlib"}
            if isinstance(node,(ast.Assign,ast.AnnAssign)) and isinstance(node.value,ast.Constant) and isinstance(node.value.value,str):assert not node.value.value.startswith("../")
            if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr in {"read_csv","to_csv"} and node.args:
                assert not isinstance(node.args[0],ast.Name),"I/O address variables are not used in new exports"
check_delivery("new_export_code_inline_relative_no_Path",True)
check_delivery("all_export_sections_executed",all(any(c["cell_type"]=="markdown" and "2026-10-30交付补齐" in "".join(c["source"]) and book["cells"][i+1].get("execution_count") for i,c in enumerate(book["cells"][:-1])) for book in documents))
pd.DataFrame(checks).to_csv("../results/workspace_audit_20261030/delivery_validation.tsv",sep="\t",index=False)
print("PASS:",len(checks),"delivery checks; exact375-sample counts,120/117 coverage,original GSEA statistics and150 PDF stars.")


PASS: 31 delivery checks; exact375-sample counts,120/117 coverage,original GSEA statistics and150 PDF stars.
